In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch
import os
import scanpy as sc
from tqdm import tqdm

# 绘图风格设置
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
sns.set_style("whitegrid")
# 忽略警告
import warnings
warnings.filterwarnings('ignore')


In [ ]:
adata = sc.read_h5ad('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib_age_normal.h5ad')  # 用 normal 数据
adata.shape


In [ ]:
adata.obs['Disease_Status'].unique(), len(adata.obs['sample'].unique()),adata.obs['Age_group'].unique()


In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
from tqdm import tqdm
import scanpy as sc
from scipy.sparse import issparse, csr_matrix, csc_matrix 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import LayerNorm
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import ElasticNetCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression, mutual_info_regression
from statsmodels.stats.multitest import multipletests
from scipy.stats import pearsonr, spearmanr, t
from scipy.special import softmax, expit
import matplotlib.pyplot as plt
import pickle
import scipy.stats as stats
import seaborn as sns
import scipy.sparse as sp
import time
import gc
import sys
import warnings
import os  

warnings.filterwarnings('ignore')


# -------------------------- 全局变量初始化 --------------------------

BASE_DATA_PATH = 'external_inputs/data'
BASE_MODEL_PATH = 'external_inputs/models'
BASE_PLOT_PATH = 'external_inputs/plots'
os.makedirs(BASE_DATA_PATH, exist_ok=True)
os.makedirs(BASE_MODEL_PATH, exist_ok=True)
os.makedirs(BASE_PLOT_PATH, exist_ok=True)
print(f"✅ 已创建目录：\n  数据目录：{BASE_DATA_PATH}\n  模型/权重目录：{BASE_MODEL_PATH}\n  图目录：{BASE_PLOT_PATH}")
ITER_ROUNDS = 6               # 迭代轮次
EMBED_DIM = 512               # 嵌入维度
EPOCHS = 20                   # 初始训练轮数
BATCH_SIZE = 6                # 批次大小
LAYER_NAME = 'bin_hv5000'     # 数据层名称
AGE_GROUP_COL = "Age_group"   # 年龄分组列名
AGE_RANGE_FIXED = (0, 100)     # 固定年龄范围
AGE_MAX_SCALE = AGE_RANGE_FIXED[1]  # 统一：年龄缩放最大值=固定范围的上限

# -------------------------- 数据预处理 --------------------------
def filter_healthy_samples(adata, disease_labels=['healthy'], healthy_save_path=None):
    """保留指定疾病类型的细胞"""
    if healthy_save_path is None:
        healthy_save_path = os.path.join(BASE_DATA_PATH, 'Endo_healthy.h5ad')
    mask = adata.obs["Disease_Status"].isin(disease_labels)
    print(f"保留健康样本数：保留{mask.sum()}个细胞（原始总数{adata.shape[0]}）")
    adata_healthy = adata[mask].copy()  # 深拷贝避免关联原始数据
    # 移除raw
    if adata_healthy.raw is not None:
        adata_healthy.raw = None
        print(f"⚠️  已移除adata_healthy.raw")
    return adata_healthy

def filter_age_group(adata, keep_groups=['A','B','C','D','E','F','G','H']):
    """选取指定Age_group"""
    mask = adata.obs[AGE_GROUP_COL].isin(keep_groups)
    print(f"掩I组过滤：保留{mask.sum()}个细胞（原始总数{adata.shape[0]}）")
    adata_filtered = adata[mask].copy()
    
    # 移除raw
    if adata_filtered.raw is not None:
        adata_filtered.raw = None
        print(f"⚠️  已移除adata_filtered.raw")
    return adata_filtered

def build_sample_celltype_mean_adata(adata, age_group_col, sample_col='sample', celltype_col='Celltype', age_col='Age'):
    """
    构建样本级别的表达矩阵（行=样本，列=细胞类型-基因均值）
    值：该样本下对应细胞类型的该基因表达均值
    obs：存储样本的细胞类型比例、年龄信息
    """
    print(f"\n=== 构建样本级表达矩阵（行=样本，列=细胞类型-基因） ===")
    # 提取基础信息
    all_samples = adata.obs[sample_col].unique()  # 所有样本（作为新adata的行）
    all_celltypes = adata.obs[celltype_col].unique()  # 所有细胞类型
    all_genes = adata.var.index.tolist()  # 所有基因
    n_samples = len(all_samples)
    n_celltypes = len(all_celltypes)
    n_genes = len(all_genes)
    # 构建列名：细胞类型_基因名（如CelltypeA_Gene1）
    new_var_names = [f"{ct}_{gene}" for ct in all_celltypes for gene in all_genes]
    n_total_cols = len(new_var_names)
    # 表达矩阵：行=样本，列=细胞类型-基因
    expr_matrix = np.zeros((n_samples, n_total_cols), dtype=np.float32)
    # 样本级obs信息（存储细胞类型比例、年龄等）
    obs_data = []
    # 细胞类型比例存储字典
    sample_ct_ratio = {}
    # 遍历每个样本计算表达均值和细胞类型比例
    for sample_idx, sample in enumerate(tqdm(all_samples, desc="计算样本级表达均值")): 
        # 筛选当前样本的所有细胞
        sample_mask = adata.obs[sample_col] == sample
        adata_sample = adata[sample_mask]
        total_cells_sample = adata_sample.n_obs  # 样本总细胞数
        # 初始化当前样本的细胞类型比例字典
        ct_ratio_dict = {}
        # 初始化当前样本的表达向量
        sample_expr = np.zeros(n_total_cols, dtype=np.float32)
        # 遍历每个细胞类型并计算均值
        for ct in all_celltypes:
            # 筛选当前样本下该细胞类型的细胞
            ct_mask = adata_sample.obs[celltype_col] == ct
            adata_sample_ct = adata_sample[ct_mask]
            ct_cell_count = adata_sample_ct.n_obs
            # 计算细胞类型比例
            ct_ratio = ct_cell_count / total_cells_sample if total_cells_sample > 0 else 0.0
            ct_ratio_dict[f"{ct}_ratio"] = ct_ratio            
            # 计算该细胞类型的基因表达均值
            if ct_cell_count > 0:
                if issparse(adata_sample_ct.X):
                    ct_gene_mean = adata_sample_ct.X.mean(axis=0).A.flatten()  # 稀疏矩阵均值
                else:
                    ct_gene_mean = adata_sample_ct.X.mean(axis=0).astype(np.float32)
            else:
                ct_gene_mean = np.zeros(n_genes, dtype=np.float32)  # 无该细胞类型则填充0
            # 将该细胞类型的基因均值填入样本表达向量的对应位置
            ct_gene_start_idx = all_celltypes.tolist().index(ct) * n_genes
            ct_gene_end_idx = ct_gene_start_idx + n_genes
            sample_expr[ct_gene_start_idx:ct_gene_end_idx] = ct_gene_mean
        # 将当前样本的表达向量填入矩阵
        expr_matrix[sample_idx] = sample_expr
        # 收集样本的obs信息（年龄+细胞类型比例）
        # 提取年龄信息（取样本内第一个非空值）
        sample_age = adata_sample.obs[age_col].dropna().iloc[0] if len(adata_sample.obs[age_col].dropna()) > 0 else np.nan
        sample_age_group = adata_sample.obs[age_group_col].dropna().iloc[0] if len(adata_sample.obs[age_group_col].dropna()) > 0 else np.nan
        # 合并年龄信息和细胞类型比例
        obs_dict = {
            'sample': sample,
            'age_original': sample_age,
            'age_group_col': sample_age_group,
            'total_cells': total_cells_sample  # 样本总细胞数
        }
        obs_dict.update(ct_ratio_dict)  # 加入所有细胞类型的比例
        obs_data.append(obs_dict) 
    # 构建var表（列注释：细胞类型+基因名）
    var_df = pd.DataFrame(index=new_var_names)
    # 拆分细胞类型和基因名，方便后续分析
    var_df['Celltype'] = [name.split('_', 1)[0] for name in new_var_names] 
    var_df['gene'] = [name.split('_', 1)[1] for name in new_var_names]
    # 构建obs表（行注释：样本信息+细胞类型比例）
    obs_df = pd.DataFrame(obs_data, index=all_samples) 
    # 初始化新adata
    new_adata = ad.AnnData(
        X=expr_matrix,
        obs=obs_df,
        var=var_df)
    # 年龄数值赋值
    print(f"\n=== 年龄数值赋值 ===")
    # 直接使用原始数值年龄
    new_adata.obs['age_numeric'] = new_adata.obs['age_original']
    # 处理缺失值
    a_group_mask = new_adata.obs[age_group_col] == "A"
    non_a_mask = ~a_group_mask
    # A组缺失值填充0
    new_adata.obs.loc[a_group_mask, 'age_numeric'] = new_adata.obs.loc[a_group_mask, 'age_numeric'].fillna(0)
    # 非A组缺失值用中位数填充
    non_a_age_vals = new_adata.obs.loc[non_a_mask, 'age_numeric']
    non_a_median = non_a_age_vals.median() if not non_a_age_vals.isna().all() else 0
    new_adata.obs.loc[non_a_mask, 'age_numeric'] = non_a_age_vals.fillna(non_a_median)
    # 打印年龄范围
    a_age_vals = new_adata.obs.loc[a_group_mask, 'age_numeric']
    non_a_age_vals_filled = new_adata.obs.loc[non_a_mask, 'age_numeric']
    return new_adata  


####  filter_low_quality_samples 这里 min_celltypes变成 3, min_cells_per_sample 变成 5
def filter_low_quality_samples(adata, sample_col='sample', min_celltypes_per_sample=3, 
                               min_cells_per_sample=5, save_path=None):
    """
    移除低质量样本（适配样本级adata的结构）
    筛选条件：
    1. 样本的细胞类型数 ≥ min_celltypes_per_sample（通过比例>0的细胞类型列统计）
    2. 样本的总细胞数 ≥ min_cells_per_sample（直接用obs的total_cells列）
    """
    # 1. 提取样本级adata的obs数据
    obs_df = adata.obs.copy()
    # 2. 统计每个样本的有效细胞类型数（比例>0的细胞类型）
    # 筛选所有细胞类型比例列（以"_ratio"结尾）
    ct_ratio_cols = [col for col in obs_df.columns if col.endswith('_ratio')]
    if len(ct_ratio_cols) == 0:
        raise ValueError(f"❌ adata.obs中无细胞类型比例列（以_ratio结尾）")
    # 计算每个样本的细胞类型数（比例>0的列数）
    obs_df['n_celltypes'] = (obs_df[ct_ratio_cols] > 0).sum(axis=1)
    # 3. 提取样本总细胞数（已存在于obs的total_cells列）
    obs_df['n_cells'] = obs_df['total_cells']
    # 4. 筛选高质量样本
    high_quality_mask = (
        (obs_df['n_celltypes'] >= min_celltypes_per_sample) &
        (obs_df['n_cells'] >= min_cells_per_sample)
    )
    high_quality_samples = obs_df[high_quality_mask].index.tolist()
    print(f"原始样本数：{len(obs_df)} | 高质量样本数：{len(high_quality_samples)}")
    print(f"过滤掉的低质量样本数：{len(obs_df) - len(high_quality_samples)}")
    # 5. 过滤adata
    adata_high_quality = adata[adata.obs.index.isin(high_quality_samples)].copy()
    return adata_high_quality


def split_sample_train_test(adata, age_group_col, sample_col='sample', test_size=0.2, val_size=0.1, random_state=42):
    """
    按【age_group分层 + 样本级别】划分训练/验证/测试集
    保证每个年龄组在三个数据集中的分布比例与原始数据完全一致
    总比例：训练集70% | 验证集10% | 测试集20%
    """
    print(f"\n=== 样本级别分层划分训练/验证/测试集（按{age_group_col}比例）===")
    # ============== 1. 构建【样本-年龄组】映射表 ==============
    # 提取所有唯一样本 + 每个样本对应的年龄组
    sample_info = adata.obs[[sample_col, age_group_col]].drop_duplicates()
    all_samples = sample_info[sample_col].values  # 所有唯一样本
    sample_stratify = sample_info[age_group_col].values  # 每个样本对应的分层标签
    # 打印原始分布
    original_counts = sample_info[age_group_col].value_counts()
    print(f"原始各年龄组样本数：\n{original_counts}")
    print(f"总样本数：{len(all_samples)}")
    # ============== 2. 分层划分数据集（两步划分，保证比例）==============
    # 第一步：分离 测试集 (test_size)
    train_val_samples, test_samples, train_val_strat, test_strat = train_test_split(
        all_samples, sample_stratify,
        test_size=test_size,
        stratify=sample_stratify,  # 分层核心参数
        random_state=random_state
    )
    # 第二步：从剩余数据中分离 训练集 和 验证集
    # 验证集占总数据的 val_size，因此在训练+验证集中的比例为 val_size/(1-test_size)
    val_ratio_in_trainval = val_size / (1 - test_size)
    train_samples, val_samples, train_strat, val_strat = train_test_split(
        train_val_samples, train_val_strat,
        test_size=val_ratio_in_trainval,
        stratify=train_val_strat,  # 再次分层，保证比例
        random_state=random_state
    )
    # ============== 3. 生成掩码并切片adata ==============
    train_mask = adata.obs[sample_col].isin(train_samples)
    val_mask = adata.obs[sample_col].isin(val_samples)
    test_mask = adata.obs[sample_col].isin(test_samples)
    adata_train = adata[train_mask].copy()
    adata_val = adata[val_mask].copy() 
    adata_test = adata[test_mask].copy()
    # ============== 4. 打印结果统计 ==============
    print(f"\n划分结果：")
    print(f"训练集样本数：{len(train_samples)} | 细胞数：{adata_train.shape[0]}")
    print(f"验证集样本数：{len(val_samples)} | 细胞数：{adata_val.shape[0]}")
    print(f"测试集样本数：{len(test_samples)} | 细胞数：{adata_test.shape[0]}")  
    # 打印训练集的年龄组分布，验证分层效果
    train_counts = adata_train.obs[age_group_col].value_counts()
    print(f"\n训练集各年龄组细胞数：\n{train_counts}")
    return adata_train, adata_val, adata_test


def get_age_group_degs(adata, age_group_col, n_top_degs=5000, deg_save_path=None,
                       age_numeric_col='age_numeric', corr_threshold=0.1,
                       mi_threshold=0.05, var_quantile=0.2, non0_threshold = 1): 
    """adata的DEG筛选"""
    if deg_save_path is None:
        deg_save_path = os.path.join(BASE_DATA_PATH, 'Endo_healthy_DEG5000_noI.h5ad')
    print(f"=== 筛选Age_group间DEG（目标{ n_top_degs }个）===")
    print(f"=== 筛选与年龄数值相关的基因（目标{n_top_degs}个）===")
    print(f"📌 年龄数值列：{age_numeric_col}，相关阈值≥{corr_threshold}，互信息阈值≥{mi_threshold}，方差分位数≥{var_quantile}")
    # ============= 1. 非零样本过滤（降低阈值，适配样本级） =============
    print(f"📌 样本级适配：非0样本数过滤阈值设为 {non0_threshold}")
    gene_nonzero_count = np.array(adata.X.astype(bool).sum(axis=0)).flatten()
    gene_nonzero_df = pd.DataFrame({
        "gene": adata.var.index,
        "nonzero_count": gene_nonzero_count
    })
    high_expr_genes = gene_nonzero_df[gene_nonzero_df["nonzero_count"] > non0_threshold]["gene"].tolist()
    print(f"非0样本数过滤：原始基因数{len(adata.var)} | 过滤后{len(high_expr_genes)}个（非0样本数>{non0_threshold}）")
    if len(high_expr_genes) == 0:
        raise ValueError("所有基因均为全0表达，无法进行后续分析")
    # ============= 2. 在过滤后的基因上计算多维度指标 =============
    # 构建高表达基因子集
    adata_high = adata[:, high_expr_genes].copy()
    # 提取表达矩阵和年龄数值
    X = adata_high.X.toarray() if issparse(adata_high.X) else adata_high.X
    age_vals = adata_high.obs[age_numeric_col].values
    # 2.1 皮尔逊相关系数（绝对值）
    from scipy.stats import pearsonr
    corr_scores = np.array([abs(pearsonr(X[:, i], age_vals)[0]) for i in range(X.shape[1])])
    corr_scores = np.nan_to_num(corr_scores, 0)
    # 2.2 互信息
    from sklearn.feature_selection import mutual_info_regression
    mi_scores = mutual_info_regression(X, age_vals, random_state=42)
    # 2.3 方差
    var_scores = np.var(X, axis=0)
    var_threshold = np.quantile(var_scores, var_quantile)
    # 多维度过滤
    filter_mask = (
        (corr_scores >= corr_threshold) & 
        (mi_scores >= mi_threshold) & 
        (var_scores >= var_threshold)
    )
    candidate_genes = adata_high.var.index[filter_mask].tolist()
    print(f"多维度筛选：{len(candidate_genes)}个基因满足相关、互信息、方差联合阈值")
    final_genes = []
    if len(candidate_genes) == 0:
        print(f"⚠️ 未筛选到满足阈值的基因，直接筛选{n_top_degs}个高表达高变基因兜底")
        sc.pp.highly_variable_genes(adata_high, n_top_genes=n_top_degs, flavor="seurat", inplace=True)
        hvg_high_expr = adata_high.var[adata_high.var["highly_variable"]].index.tolist()
        final_genes = hvg_high_expr[:n_top_degs]
    elif len(candidate_genes) < n_top_degs:
        need_hvgs = n_top_degs - len(candidate_genes)
        print(f"候选基因数量不足（{len(candidate_genes)}<{n_top_degs}），补充{need_hvgs}个高表达HVG")
        sc.pp.highly_variable_genes(adata_high, n_top_genes=need_hvgs*2, flavor="seurat", inplace=True)
        hvgs = adata_high.var[adata_high.var["highly_variable"]].index.tolist()
        final_genes = candidate_genes + [g for g in hvgs if g not in candidate_genes][:need_hvgs]
    else:
        # 候选基因充足，取前n_top_degs（按加权得分排序，保证选取最相关的）
        weighted_scores = 0.6 * corr_scores + 0.4 * mi_scores
        sorted_idx = np.argsort(weighted_scores)[::-1]  # 降序
        sorted_genes = adata_high.var.index[sorted_idx].tolist()
        final_genes = [g for g in sorted_genes if g in candidate_genes][:n_top_degs]
    if len(final_genes) == 0:
        print(f"❌ 筛选后仍无有效基因，直接取前{n_top_degs}个非0基因")
        final_genes = gene_nonzero_df[gene_nonzero_df["nonzero_count"] > 0]["gene"].tolist()[:n_top_degs]
    print(f"✅ 最终基因集大小：{len(final_genes)}（目标{n_top_degs}）")
    # ============= 5. 构建DEG子adata并校验无NaN/Inf =============
    print(f"\n=== 构建DEG子adata并校验无NaN/Inf ===")
    gene_mask = adata.var.index.isin(final_genes)
    adata_deg_only = adata[:, gene_mask].copy()
    # 校验异常值
    if sp.issparse(adata_deg_only.X):
        nan_count = np.isnan(adata_deg_only.X.data).sum()
        inf_count = np.isinf(adata_deg_only.X.data).sum()
    else:
        nan_count = np.isnan(adata_deg_only.X).sum()
        inf_count = np.isinf(adata_deg_only.X).sum()
    if nan_count > 0 or inf_count > 0:
        print(f"❌ DEG子adata存在异常值：NaN={nan_count} | Inf={inf_count}，程序终止")
        sys.exit(1)
    print(f"✅ DEG子adata无NaN/Inf，维度：{adata_deg_only.shape}")
    return adata_deg_only


def bin_expression_matrix(adata_deg_only, layer_name: str = LAYER_NAME, 
                          bin_save_path=None, n_bins=32):
    """
    1. 对adata.X（样本×细胞类型-基因）全局分n_bins（所有值参与）
    2. 对adata.obs中的celltype_ratio列分bin（0值单独保留，非0值分n_bins档）
    3. 合并结果：将ratio分bin结果拼接到原分bin矩阵后，存入指定layer
    """
    if bin_save_path is None:
        bin_save_path = os.path.join(BASE_DATA_PATH, 'Endo_healthy_bin_noI.h5ad')
    # ===================== 第一步：对adata.X分bin =====================
    print(f"=== 开始全局分bin流程（所有值参与） ===")
    n_samples, n_expr_features = adata_deg_only.shape
    print(f"DEG子adata信息：样本数={n_samples} | 表达特征数={n_expr_features}")
    # 1. 处理表达矩阵X
    X_target = adata_deg_only.X
    if sp.issparse(X_target):
        print(f"将稀疏矩阵转换为稠密数组...")
        # 分块转换避免内存溢出
        try:
            X_target = X_target.toarray()
        except MemoryError:
            X_target = np.zeros((n_samples, n_expr_features), dtype=np.float32)
            chunk_size = 5000
            for i in range(0, n_expr_features, chunk_size):
                end = min(i + chunk_size, n_expr_features)
                X_target[:, i:end] = adata_deg_only.X[:, i:end].toarray()
    X_target = X_target.astype(np.float32)
    print(f"表达矩阵（稠密）：{X_target.shape} | 数据类型：{X_target.dtype}")
    # 检查并清理表达矩阵的NaN/Inf
    check_nan_inf(X_target, "X_target", context="表达矩阵预处理后")
    X_clean = np.nan_to_num(X_target, nan=0.0, posinf=0.0, neginf=0.0)
    all_x_values = X_clean.flatten()
    x_valid_count = np.count_nonzero(all_x_values)
    x_total_count = len(all_x_values)
    x_zero_ratio = (x_total_count - x_valid_count) / x_total_count * 100 if x_total_count > 0 else 100
    print(f"✅ 表达矩阵统计：总元素={x_total_count} | 非0有效值={x_valid_count} | 零值占比={x_zero_ratio:.2f}%")
    # 2. 处理细胞类型比例列
    ratio_cols = [col for col in adata_deg_only.obs.columns if col.endswith('_ratio')]
    if not ratio_cols:
        raise ValueError(f"❌ adata.obs中未找到以'_ratio'结尾的细胞类型比例列") 
    n_ratio_features = len(ratio_cols)
    print(f"\n找到细胞类型比例列：{ratio_cols}（共{n_ratio_features}列）")
    # 检查并清理比例矩阵的NaN/Inf
    ratio_matrix = adata_deg_only.obs[ratio_cols].values.astype(np.float32)
    check_nan_inf(ratio_matrix, "ratio_matrix", context="比例矩阵预处理后")
    ratio_clean = np.nan_to_num(ratio_matrix, nan=0.0, posinf=0.0, neginf=0.0)
    all_ratio_values = ratio_clean.flatten()
    ratio_valid_count = np.count_nonzero(all_ratio_values)
    ratio_total_count = len(all_ratio_values)
    ratio_zero_ratio = (ratio_total_count - ratio_valid_count) / ratio_total_count * 100 if ratio_total_count > 0 else 100
    print(f"✅ 比例矩阵统计：总元素={ratio_total_count} | 非0有效值={ratio_valid_count} | 零值占比={ratio_zero_ratio:.2f}%")
    # ===================== 第二步：分bin函数（区分表达矩阵/比例矩阵） =====================
    def safe_qcut_expr(arr, n_bins=n_bins, desc="表达矩阵"):
        """表达矩阵全局分bin（兼容全0/有效值不足）"""
        unique_vals = np.unique(arr)
        if len(unique_vals) < 2:
            print(f"⚠️ {desc} 唯一值数量<2（仅{unique_vals}），无法分bin，返回全0矩阵")
            return np.zeros_like(arr, dtype=np.uint8), None  
        try:
            qcut_result, bin_edges = pd.qcut(
                arr, q=n_bins, retbins=True, duplicates="drop"
            )
            bin_labels = pd.factorize(qcut_result)[0] + 1  # 标签从1开始
            bin_labels = np.clip(bin_labels, 1, n_bins).astype(np.uint8)
            print(f"✅ {desc} 分bin完成：实际分箱数={len(bin_edges)-1} | 分bin值范围=[{bin_labels.min()}, {bin_labels.max()}]")
            return bin_labels, bin_edges
        except Exception as e:
            print(f"❌ {desc} 分bin失败：{e}，返回全0矩阵")
            return np.zeros_like(arr, dtype=np.uint8), None
    def safe_qcut_ratio(arr, n_bins=n_bins, desc="细胞类型比例"):
        """比例矩阵分bin（0值单独保留，非0值分n_bins档）"""
        # 初始化结果矩阵为全0
        bin_labels = np.zeros_like(arr, dtype=np.uint8)
        # 分离0值和非0值的索引
        zero_mask = arr == 0
        non_zero_vals = arr[~zero_mask]
        if len(non_zero_vals) == 0:
            print(f"⚠️ {desc} 无有效非0值，返回全0矩阵")
            return bin_labels, None    
        # 仅对非0值进行分bin
        unique_non_zero = np.unique(non_zero_vals)
        if len(unique_non_zero) < 2:
            print(f"⚠️ {desc} 非0唯一值数量<2（仅{unique_non_zero}），非0值统一标记为1")
            bin_labels[~zero_mask] = 1
        else:
            try:
                # 对非0值分n_bins档，标签从1到n_bins
                qcut_result, bin_edges = pd.qcut(
                    non_zero_vals, q=n_bins, retbins=True, duplicates="drop"
                )
                non_zero_labels = pd.factorize(qcut_result)[0] + 1  # 标签从1开始
                # 确保最大值不超过n_bins（兼容duplicates="drop"导致的分箱数减少）
                non_zero_labels = np.clip(non_zero_labels, 1, n_bins).astype(np.uint8)
                # 将非0值的分bin结果赋值回原矩阵
                bin_labels[~zero_mask] = non_zero_labels
                print(f"✅ {desc} 分bin完成：非0值实际分箱数={len(bin_edges)-1} | 分bin值范围=[0, {bin_labels.max()}]")
                return bin_labels, bin_edges
            except Exception as e:
                print(f"❌ {desc} 非0值分bin失败：{e}，非0值统一标记为1")
                bin_labels[~zero_mask] = 1
        return bin_labels, None
    # 1. 表达矩阵分bin
    x_flat_binned, x_bin_edges = safe_qcut_expr(all_x_values, n_bins=n_bins, desc="表达矩阵")
    x_binned_matrix = x_flat_binned.reshape((n_samples, n_expr_features))
    # 2. 比例矩阵分bin（0值保留，非0值分bin）
    ratio_flat_binned, ratio_bin_edges = safe_qcut_ratio(all_ratio_values, n_bins=n_bins, desc="细胞类型比例")
    ratio_binned_matrix = ratio_flat_binned.reshape((n_samples, n_ratio_features))
    # ===================== 第三步：合并结果并构建adata =====================
    print(f"\n=== 合并分bin结果 ===")
    # 合并分bin矩阵
    combined_binned_matrix = np.hstack([x_binned_matrix, ratio_binned_matrix])
    n_combined_features = combined_binned_matrix.shape[1]
    print(f"合并后分bin矩阵维度：{combined_binned_matrix.shape}")
    print(f"合并后分bin值范围：[{combined_binned_matrix.min()}, {combined_binned_matrix.max()}]")
    # 构建合并后的var
    expr_var = adata_deg_only.var.copy()
    expr_var['feature_type'] = 'Expression'
    ratio_var = pd.DataFrame({
        'Celltype': [col.replace('_ratio', '') for col in ratio_cols],
        'gene': np.nan,
        'feature_type': 'Celltype_ratio'
    }, index=ratio_cols)
    combined_var = pd.concat([expr_var, ratio_var], axis=0)
    assert len(combined_var) == n_combined_features, "var维度不匹配"
    # 构建最终adata
    adata_combined = ad.AnnData(
        X=combined_binned_matrix,
        obs=adata_deg_only.obs.copy(),
        var=combined_var,
        uns=adata_deg_only.uns.copy() if adata_deg_only.uns else {}
    )
    adata_combined.layers[layer_name] = combined_binned_matrix
    # 记录元信息
    adata_combined.uns.update({
        "x_bin_edges": x_bin_edges,
        "ratio_bin_edges": ratio_bin_edges,
        "ratio_cols": ratio_cols,
        "n_expr_features": n_expr_features,
        "n_ratio_features": n_ratio_features,
        "ratio_bin_logic": "0值保留为0，非0值分n_bins档（标签1~n_bins）"
    })
    # ===================== 第四步：保存与清理 =====================
    # 保存分bin边界
    if x_bin_edges is not None:
        np.save(os.path.join(BASE_DATA_PATH, 'healthy_x_bin_edges.npy'), x_bin_edges)
    if ratio_bin_edges is not None:
        np.save(os.path.join(BASE_DATA_PATH, 'healthy_ratio_bin_edges.npy'), ratio_bin_edges)
    # 清理内存
    del X_target, ratio_matrix, combined_binned_matrix, x_binned_matrix, ratio_binned_matrix
    gc.collect()
    return adata_combined

def get_age_stage_label_by_group(age_group_series):
    """
    基于Age_group列的类别生成阶段标签
    参数：
        age_group_series: Age_group列的数组/Series（元素为A/B/C/D/E/F/G/H）
    返回：
        stage_labels: 阶段标签（0=未衰老、1=青年、2=中年、3=老年）
    """
    # 定义分组映射规则（核心：按你的要求）
    group_to_stage = {
        'A': 0,   # 0-20岁：未衰老
        'B': 1,   # 青年
        'C': 1,
        'D': 2,   # 中年
        'E': 2,
        'F': 3,   # 老年
        'G': 3,
        'H': 3
    }
    # 向量化映射（高效处理批量数据）
    stage_labels = np.array([group_to_stage.get(str(g).strip(), 0) for g in age_group_series])
    return stage_labels



# -------------------------- 先验计算适配（调整归一化范围） --------------------------
def compute_ct_gene_age_contribution(adata, age_col='Age', layer=LAYER_NAME, 
                                     corr_threshold=0.15, std_threshold=0.1):
    """
    CT基因筛选：同时过滤相关性和标准差，只保留年龄强相关基因
    - corr_threshold：基因与年龄的皮尔逊相关系数绝对值阈值
    - std_threshold：基因贡献度的标准差阈值
    """
    import scipy.stats as stats
    # 1. 提取表达矩阵和年龄值
    age_vals = adata.obs[age_col].values
    if layer in adata.layers:
        gene_expr = adata.layers[layer].toarray() if issparse(adata.layers[layer]) else adata.layers[layer]
    else:
        gene_expr = adata.X.toarray() if issparse(adata.X) else adata.X
    # 2. 计算每个基因与年龄的皮尔逊相关系数
    gene_corr = []
    for i in range(gene_expr.shape[1]):
        corr, _ = stats.pearsonr(gene_expr[:, i], age_vals)
        gene_corr.append(abs(corr) if not np.isnan(corr) else 0)
    gene_corr = np.array(gene_corr)
    # 3. 计算每个基因的表达标准差
    gene_std = np.std(gene_expr, axis=0)
    # 4. 计算基因贡献度（相关性 * 标准差）
    gene_contribution = gene_corr * gene_std
    # 5. 构建完整的Series（索引=所有基因，值=贡献度）
    contribution_series = pd.Series(
        gene_contribution,
        index=adata.var.index,
        name='age_contribution'
    )
    # 6. 双重过滤：相关性达标 + 贡献度波动达标
    gene_mask = (gene_corr >= corr_threshold) & (gene_std >= std_threshold)
    high_contribution_genes = adata.var.index[gene_mask]
    # 输出筛选结果
    print(f"筛选前基因数：{adata.shape[1]} | 筛选后基因数：{len(high_contribution_genes)}")
    print(f"保留基因的平均年龄相关性：{np.mean(gene_corr[gene_mask]):.3f}")
    contribution_series = (contribution_series - contribution_series.min()) / (contribution_series.max() - contribution_series.min() + 1e-8)
    contribution_series = contribution_series * 0.9 + 0.1
    # 返回完整的贡献度Series（而非仅基因名），方便后续reindex
    return contribution_series


def compute_sample_level_prior_u(adata, ct_gene_contribution, age_col="Age", 
                                 tauu_factor=1.0, round_idx=0):
    """
    将tauu_schedule融入先验计算
    - tauu_factor: 先验强度因子，随训练轮次衰减
    - round_idx: 当前训练轮次，用于调整先验强度
    """
    print(f"=== 计算样本级先验u值 (轮次{round_idx}, τu因子={tauu_factor:.2f}) ===")
    X = adata.X.copy()
    if sp.issparse(X):
        X = X.toarray()
    age_numeric = adata.obs[age_col].values
    # 年龄归一化
    age_min = np.nanmin(age_numeric)
    age_max = np.nanmax(age_numeric)
    age_range = age_max - age_min + 1e-8
    age_norm = (age_numeric - age_min) / age_range
    # 特征标准化（MinMax）
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_norm = scaler.fit_transform(X)
    X_norm = np.clip(X_norm, 0.0, 1.1)
    # 贡献度处理
    if isinstance(ct_gene_contribution, pd.Series):
        contrib_array = ct_gene_contribution.reindex(adata.var.index).values
    else:
        contrib_array = ct_gene_contribution
    contrib_array = np.nan_to_num(contrib_array, nan=0.5)
    contrib_norm = contrib_array / (np.max(contrib_array) + 1e-8)
    # 计算基础先验信号（不包含tauu衰减）
    base_sample_u = []
    n_features = X_norm.shape[1]
    for i in range(adata.shape[0]):
        expr_vector = X_norm[i]
        weighted_expr = np.dot(expr_vector, contrib_norm)
        relative_signal = weighted_expr / np.sqrt(n_features)
        signal_stable = np.tanh(relative_signal)
        raw_u = age_norm[i] * signal_stable
        base_sample_u.append(raw_u)
    base_sample_u = np.array(base_sample_u, dtype=np.float64)
    sample_u = base_sample_u * tauu_factor
    print(f" 先验u值统计（应用τu衰减后）:")
    print(f" 范围：{np.min(sample_u):.3f} ~ {np.max(sample_u):.3f}")
    print(f" 均值：{np.mean(sample_u):.3f}, 标准差：{np.std(sample_u):.3f}")
    return sample_u 

def binary_nnNU_risk(g, is_non_control_mask, alpha):
    """保留：nnNU风险函数，解决inf问题"""
    if len(g.shape) > 1:
        g_binary = g.squeeze(dim=1)
    else:
        g_binary = g
    g_binary = torch.clamp(g_binary, min=-100.0, max=100.0)
    pos_mask = is_non_control_mask
    neg_mask = ~is_non_control_mask
    if pos_mask.sum() == 0:
        pos_loss = torch.tensor(0.0, device=g_binary.device)
    else:
        pos_loss = torch.nn.functional.softplus(-g_binary[pos_mask]).mean()
    if neg_mask.sum() == 0:
        neg_loss = torch.tensor(0.0, device=g_binary.device)
    else:
        neg_loss = torch.nn.functional.softplus(g_binary[neg_mask]).mean()
    risk = alpha * pos_loss + (1 - alpha) * neg_loss
    risk = torch.nan_to_num(risk, nan=0.0, posinf=10.0, neginf=0.0)
    return risk


class ResidualBlock(nn.Module):
    """残差块：强化特征提取，缓解梯度消失"""
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim * 2)
        self.bn1 = nn.BatchNorm1d(out_dim * 2)
        self.fc2 = nn.Linear(out_dim * 2, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)
        self.dropout = nn.Dropout(dropout)
        self.shortcut = nn.Sequential()
        if in_dim != out_dim:
            self.shortcut = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim)
            )
    def forward(self, x):
        residual = self.shortcut(x)
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.bn2(self.fc2(x))
        x = x + residual
        x = F.relu(x)
        return x

class GateFusion(nn.Module):
    """门控融合层：深度融合latent与先验u值"""
    def __init__(self, embed_dim):
        super().__init__()
        self.u_proj = nn.Linear(1, embed_dim)
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Sigmoid()
        )
    def forward(self, latent, u_prior):
        u_proj = F.relu(self.u_proj(u_prior.unsqueeze(1)))
        concat = torch.cat([latent, u_proj], dim=1)
        gate_weight = self.gate(concat)
        fused_latent = latent * gate_weight + u_proj * (1 - gate_weight)
        return fused_latent

class StageGatedRegressor(nn.Module):
    """阶段门控回归头：适配不同年龄阶段特征"""
    def __init__(self, embed_dim, n_stages=4):
        super().__init__()
        self.n_stages = n_stages
        self.stage_feat = nn.ModuleList()
        for i in range(n_stages):
            self.stage_feat.append(
                nn.Sequential(
                    nn.Linear(embed_dim, embed_dim//2),
                    nn.LayerNorm(embed_dim//2),
                    nn.ReLU(),
                    nn.Dropout(0.2)
                )
            )
        self.stage_reg = nn.ModuleList([
            nn.Linear(embed_dim//2, 1) for _ in range(n_stages)
        ])
        self.gate = nn.Sequential(
            nn.Linear(embed_dim, n_stages),
            nn.Softmax(dim=1)
        )
    def forward(self, x, stage_probs):
        stage_preds = []
        for i in range(self.n_stages):
            feat = self.stage_feat[i](x)
            pred = self.stage_reg[i](feat).squeeze(1)
            stage_preds.append(pred)
        stage_preds = torch.stack(stage_preds, dim=1)
        gate_weight = self.gate(x) * stage_probs
        gate_weight[:, 0] = gate_weight[:, 0] * 1.2
        gate_weight[:, 3] = gate_weight[:, 3] * 1.2
        gate_weight = gate_weight / gate_weight.sum(dim=1, keepdim=True)
        age_pred = torch.sum(gate_weight * stage_preds, dim=1)
        return age_pred

class BinarynnNUClassifier(nn.Module):
    """二分类nnNU分类器"""
    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze()

class AgeRegressionAE(nn.Module):
    """
    有监督回归型自编码器（基因权重端到端学习）
    - 输入：基因表达分箱值 x，先验 u_prior
    - 输出：潜在表示 latent，重构分位数 gene_quantile_out，年龄预测 age_pred，阶段概率 stage_probs
    """
    def __init__(self, n_genes, n_bins, embed_dim=128, age_max_train=None,
                 hidden_dims=[1024, 512], dropout=0.1, n_stages=4, 
                 prior_gene_weights=None):
        super(AgeRegressionAE, self).__init__()
        self.n_genes = n_genes
        self.n_quantiles = 9
        self.embed_dim = embed_dim
        self.age_max_train = age_max_train if age_max_train is not None else 1.0
        self.n_stages = n_stages
        # 可学习基因权重（端到端学习）
        self.gene_weight = nn.Parameter(torch.tensor(prior_gene_weights, dtype=torch.float32))
        # ---------- 编码器（带注意力） ----------
        encoder_layers = []
        prev_dim = n_genes
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.LayerNorm(h_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
            prev_dim = h_dim
        self.encoder = nn.Sequential(*encoder_layers)
        self.attention = nn.Linear(prev_dim, embed_dim)
        self.encoder_final = nn.Linear(embed_dim, embed_dim)
        # ---------- 阶段分类器 ----------
        self.stage_classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_stages),
            nn.Softmax(dim=1)
        )
        # ---------- 阶段门控回归头 ----------
        self.age_regressor = StageGatedRegressor(embed_dim, n_stages)
        # ---------- 解码器（重构分位数） ----------
        decoder_layers = []
        prev_dim = embed_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.LayerNorm(h_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
            prev_dim = h_dim 
        self.decoder = nn.Sequential(*decoder_layers)
        self.decoder_final = nn.Linear(prev_dim, n_genes * self.n_quantiles)
    def forward(self, x, u_prior=None):
        x_weighted = x * self.gene_weight
        # 编码
        h = self.encoder(x_weighted)
        att_weights = torch.softmax(self.attention(h), dim=1)
        h_att = att_weights * h
        latent = self.encoder_final(h_att)
        # 阶段分类
        stage_probs = self.stage_classifier(latent)
        # 年龄回归
        age_pred = self.age_regressor(latent, stage_probs)
        # 解码重构
        recon_h = self.decoder(latent)
        recon_flat = self.decoder_final(recon_h)
        gene_quantile_out = recon_flat.reshape(-1, self.n_genes, self.n_quantiles)
        return latent, gene_quantile_out, age_pred, stage_probs
    def predict_age(self, x, u_prior, scale_back=True):
        self.eval()
        with torch.no_grad():
            latent, _, age_pred_scaled, stage_probs = self.forward(x, u_prior)
            age_pred_scaled = torch.clamp(age_pred_scaled, min=0.0)
            if scale_back and self.age_max_train is not None:
                age_pred = age_pred_scaled * self.age_max_train
            else:
                age_pred = age_pred_scaled
        return age_pred, latent, stage_probs

class AgeRegressionDataset(Dataset):
    """数据集类（无需权重，直接使用原始分箱矩阵）"""
    def __init__(self, expr_binned, age_scaled, cell_prior_u, age_group=None):
        self.expr_binned = torch.tensor(expr_binned, dtype=torch.float32)
        self.age_scaled = torch.tensor(age_scaled, dtype=torch.float32)
        self.cell_prior_u = torch.tensor(cell_prior_u, dtype=torch.float32)
        if age_group is not None:
            self.age_group = np.array(age_group)
        else:
            self.age_group = np.zeros(len(expr_binned), dtype=str)
    def __len__(self):
        return len(self.expr_binned)
    def __getitem__(self, idx):
        return self.expr_binned[idx], self.age_scaled[idx], self.cell_prior_u[idx], self.age_group[idx]

class MultiRoundWeightProcessor:
    def __init__(self):
        self.weight_rounds = []
        self.gene_weight_rounds = []
        self.eta_schedule = {}
        self.current_round = 0
    def add_round_weights(self, eta, tauu_factor, gene_weights=None):
        self.weight_rounds.append({'eta': eta, 'tauu_factor': tauu_factor})
        self.gene_weight_rounds.append(gene_weights.copy() if gene_weights is not None else None)
        self.eta_schedule[self.current_round] = eta
        self.current_round += 1
    def apply_single_round_weighting(self, expr_binned, round_idx, n_bins=32):
        if round_idx == 0:
            return expr_binned.copy()
        prev_idx = round_idx - 1
        if prev_idx >= len(self.gene_weight_rounds) or self.gene_weight_rounds[prev_idx] is None:
            raise ValueError(f"轮次 {prev_idx} 的基因权重不存在")
        return expr_binned * self.gene_weight_rounds[prev_idx]
    def get_round_tauu_factor(self, round_idx=0):
        return self.weight_rounds[round_idx]['tauu_factor']
    def truncate_to_round(self, target_round_idx):
        if target_round_idx < 0 or target_round_idx >= len(self.weight_rounds):
            raise ValueError(f"目标轮次 {target_round_idx} 超出范围，总轮次：{len(self.weight_rounds)}")
        truncated = MultiRoundWeightProcessor()
        truncated.weight_rounds = self.weight_rounds[:target_round_idx+1]
        truncated.gene_weight_rounds = self.gene_weight_rounds[:target_round_idx+1]
        truncated.eta_schedule = {i: self.eta_schedule[idx] for i, idx in enumerate(range(target_round_idx+1))}
        truncated.current_round = len(truncated.weight_rounds)
        return truncated
    def save(self, filepath):
        with open(filepath, 'wb') as f:
            pickle.dump({
                'weight_rounds': self.weight_rounds,
                'gene_weight_rounds': self.gene_weight_rounds,
                'eta_schedule': self.eta_schedule
            }, f)
    @classmethod
    def load(cls, filepath):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        processor = cls()
        processor.weight_rounds = data['weight_rounds']
        processor.gene_weight_rounds = data.get('gene_weight_rounds', [])
        processor.eta_schedule = data['eta_schedule']
        processor.current_round = len(processor.weight_rounds)
        return processor

def check_nan_inf(x, var_name, context=""):
    if isinstance(x, torch.Tensor):
        has_nan = torch.isnan(x).any().item()
        has_inf = torch.isinf(x).any().item()
    elif isinstance(x, np.ndarray):
        has_nan = np.isnan(x).any()
        has_inf = np.isinf(x).any()
    else:
        has_nan = np.isnan(x) if np.isscalar(x) else False
        has_inf = np.isinf(x) if np.isscalar(x) else False
    if has_nan or has_inf:
        raise ValueError(
            f"\n❌ 数值异常！环节：{context}\n"
            f"变量名：{var_name}\n"
            f"是否含NaN：{has_nan} | 是否含Inf：{has_inf}\n"
            f"变量形状：{getattr(x, 'shape', 'scalar')}\n"
            f"变量示例：{x[:5] if hasattr(x, '__getitem__') else x}"
        )

def quantile_loss(y_true, y_pred, quantile_levels, gene_weights):
    """
    分位数损失，使用模型内部学习的基因权重 gene_weights
    gene_weights: 已通过 sigmoid 约束的权重，形状 (n_genes,)
    """
    y_true_clamped = torch.clamp(y_true, min=0.0, max=31.0)
    y_pred_clamped = torch.clamp(y_pred, min=0.0, max=31.0)
    y_true_quantile = y_true_clamped / 31.0
    y_pred_quantile = y_pred_clamped / 31.0
    batch_size, n_genes = y_true_quantile.shape
    w_j_expanded = gene_weights.unsqueeze(0).expand(batch_size, n_genes).unsqueeze(-1).expand(batch_size, n_genes, len(quantile_levels))
    quantile_levels_expanded = quantile_levels.unsqueeze(0).unsqueeze(0).expand(batch_size, n_genes, len(quantile_levels))
    y_true_expanded = y_true_quantile.unsqueeze(-1).expand(batch_size, n_genes, len(quantile_levels))
    errors = y_true_expanded - y_pred_quantile
    positive_errors = torch.max(errors, torch.tensor(0.0, device=errors.device))
    negative_errors_abs = torch.max(-errors, torch.tensor(0.0, device=errors.device))
    loss = quantile_levels_expanded * positive_errors + (1 - quantile_levels_expanded) * negative_errors_abs    
    loss = loss * w_j_expanded
    loss = torch.nan_to_num(loss, nan=0.0, posinf=1.0, neginf=0.0)
    w_bar = gene_weights.mean()
    loss = loss.sum() / (batch_size * n_genes * len(quantile_levels))
    return loss / w_bar
    

def age_regression_loss(y_true_scaled, y_pred_scaled, stage_true, stage_probs, age_max_train=1.0):
    mae_loss = F.l1_loss(y_pred_scaled, y_true_scaled)
    stage_loss = F.cross_entropy(stage_probs, stage_true.long())
    total_loss = 0.9 * mae_loss + 0.1 * stage_loss
    print(f"MAE损失：{mae_loss.item():.4f} | 阶段损失：{0.1 * stage_loss.item():.4f}")
    return total_loss, mae_loss, stage_loss

def get_age_stage_label_by_group(age_group_batch):
    """
    将年龄组标签（如 'A','B',...）映射为阶段标签（0,1,2,3）
    此处简单映射：A->0, B->1, C->2, D->3, 其余也映射为 0-3 中的一个（可根据实际修改）
    """
    mapping = {'A':0, 'B':1, 'C':2, 'D':3, 'E':0, 'F':1, 'G':2, 'H':3}  # 示例
    return [mapping.get(g, 0) for g in age_group_batch]

def u_prior_regularization_loss(latent, u_prior, device):
    """先验u值正则化：迫使latent与u_prior相关"""
    latent_mean = latent.mean(dim=1)
    u_mean = u_prior
    # 计算余弦相似度，最大化相似度
    sim = F.cosine_similarity(latent_mean.unsqueeze(1), u_mean.unsqueeze(1), dim=1).squeeze()
    loss = 1 - sim.mean()
    return loss

def load_model_params_safely(model, prev_state_dict, device):
    """安全加载模型参数"""
    model_dict = model.state_dict()
    loadable_layers = []
    for name, param in prev_state_dict.items():
        if name in model_dict and param.shape == model_dict[name].shape:
            loadable_layers.append(name)
            model_dict[name] = param
    model.load_state_dict(model_dict)
    print(f"✅ 安全加载{len(loadable_layers)}/{len(model_dict)}个参数")
    return model



def train_age_regression_ae(adata_train, adata_val, embed_dim=128, epochs=20, iter_rounds=6, batch_size=32, n_bins=32,
                            layer_name=LAYER_NAME, celltype_col='Celltype', age_group_col=AGE_GROUP_COL, 
                            var_threshold=1e-5, select_k=1500, age_fuse_method="median", age_range_fixed=None):
    for adata in [adata_train, adata_val]:
        if "age_numeric" not in adata.obs.columns:
            raise ValueError(f"❌ adata.obs中无age_numeric列，请先运行build_sample_celltype_mean_adata")
    # 年龄缩放（0-1）
    print(f"\n=== 年龄数据缩放处理 ===")
    if age_range_fixed is not None:
        age_max_scale = age_range_fixed[1]
        print(f"使用手动指定的年龄最大值：{age_max_scale}")
    else:
        age_max_scale = np.nanmax(adata_train.obs["age_numeric"].values)
        print(f"使用训练集年龄最大值：{age_max_scale}")
    adata_train.obs["age_scaled"] = adata_train.obs["age_numeric"] / age_max_scale
    adata_val.obs["age_scaled"] = adata_val.obs["age_numeric"] / age_max_scale
    print(f"训练集缩放后年龄范围：[{adata_train.obs['age_scaled'].min():.4f}, {adata_train.obs['age_scaled'].max():.4f}]")
    print(f"验证集缩放后年龄范围：[{adata_val.obs['age_scaled'].min():.4f}, {adata_val.obs['age_scaled'].max():.4f}]")    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # 分离非A组数据（AE训练仅用非A组）
    non_a_mask_train = adata_train.obs[age_group_col] != 'A'
    adata_train_non_a = adata_train[non_a_mask_train].copy()
    non_a_mask_val = adata_val.obs[age_group_col] != 'A'
    adata_val_non_a = adata_val[non_a_mask_val].copy()
    print(f"\n=== 模型训练初始化 ===")
    print(f"训练集（非A组）样本数：{adata_train_non_a.shape[0]}")
    print(f"验证集（非A组）样本数：{adata_val_non_a.shape[0]}")
    # 提取缩放后的年龄
    print(f"\n=== 年龄回归配置 ===")
    train_age_scaled = adata_train_non_a.obs["age_scaled"].values
    val_age_scaled = adata_val_non_a.obs["age_scaled"].values
    print(f"训练集缩放年龄范围：{train_age_scaled.min():.4f} ~ {train_age_scaled.max():.4f}")
    print(f"验证集缩放年龄范围：{val_age_scaled.min():.4f} ~ {val_age_scaled.max():.4f}")
    # 定义分位数水平
    quantile_levels = torch.tensor([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99], device=DEVICE)
    # 初始化模型
    model = None
    prev_state_dict_path = None
    # 调度策略
    eta_schedule =  {0:2, 1:5, 2:8, 3:10, 4:15, 5:20}  # 可改
    tauu_schedule = {0:0.6, 1:0.5, 2:0.4, 3:0.3, 4:0.2, 5:0.1}  # 先验的重要性
    print(f"✅ 平缓eta调度：{eta_schedule}")
    try:
        # 数据校验
        required_cols = ["age_numeric", "age_scaled", age_group_col]
        missing_cols = [col for col in required_cols if col not in adata_train_non_a.obs.columns]
        if missing_cols:
            raise ValueError(f"❌ 训练集非A组缺失必要列：{missing_cols}")
        if layer_name not in adata_train_non_a.layers:
            raise KeyError(f"❌ 训练集非A组layers中无{layer_name}")
        # 基础参数：直接使用原始分箱矩阵，不再计算外部权重
        expr_binned_train = adata_train_non_a.layers[layer_name].copy()
        expr_binned_val = adata_val_non_a.layers[layer_name].copy()
        check_nan_inf(expr_binned_train, "expr_binned_train", "训练集非A组分bin矩阵")
        check_nan_inf(expr_binned_val, "expr_binned_val", "验证集非A组分bin矩阵")
        # ========== 计算ct_genes年龄贡献度 + 样本级先验u值 ==========
        # 1. 计算ct_genes对年龄的贡献度（ElasticNet）
        ct_gene_contribution = compute_ct_gene_age_contribution(adata_train_non_a, "age_numeric")
        # 保存贡献度
        ct_gene_contribution_path = os.path.join(BASE_MODEL_PATH, "ct_gene_contribution.pkl")
        ct_gene_contribution.to_pickle(ct_gene_contribution_path)
        print(f"✅ 保存贡献度到：{ct_gene_contribution_path}")
        # 2. 计算样本级先验u值（初始轮次）
        cell_prior_u_train = compute_sample_level_prior_u(
            adata_train_non_a, ct_gene_contribution, "age_numeric", 
            tauu_factor=0.6, round_idx=0
        )
        cell_prior_u_val = compute_sample_level_prior_u(
            adata_val_non_a, ct_gene_contribution, "age_numeric",
            tauu_factor=0.6, round_idx=0
        )
        check_nan_inf(cell_prior_u_train, "cell_prior_u_train", "训练集非A组样本级先验u值")
        check_nan_inf(cell_prior_u_val, "cell_prior_u_val", "验证集非A组样本级先验u值")
        # nnNU参数（保留）
        nnnu_params = {
            "hidden_dim":16, "dropout":0.5, "alpha_start":0.9, "alpha_decrement":(0.9 - 0.1) / iter_rounds,
            "calibration_target":0.15, "train_rounds":200,
            "lr":1e-4, "weight_decay":5e-4
        }
        # 跟踪全局最佳轮次
        best_global_round_idx = None
        best_global_val_loss = float('inf')
        best_global_model_path = None
        best_val_total_loss = float('inf')
        generated_pth_files = []
        # 初始化多轮权重处理器（仅记录eta和tauu）
        weight_processor = MultiRoundWeightProcessor()
        # 保存上一轮的基因权重，用于下一轮数据加权
        prev_gene_weights = None  # 初始化为None，第一轮无权重
        for round_idx in tqdm(range(iter_rounds), desc="整体迭代轮次"):
            DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            # 设置上一轮检查点
            if round_idx > 0:
                prev_state_dict_path = os.path.join(BASE_MODEL_PATH, f"age_regression_checkpoint_round_{round_idx-1}.pth") 
                if 'model' in locals():
                    del model
                if 'optimizer' in locals():
                    del optimizer
                if 'scheduler' in locals():
                    del scheduler
                gc.collect()
                torch.cuda.empty_cache()
                time.sleep(0.5)
                free_mem, total_mem = torch.cuda.mem_get_info(0)
                print(f"✅ 已清理上一轮({round_idx-1})显存，当前GPU可用显存：{free_mem/1024**3:.2f} GiB")
            current_eta = eta_schedule.get(round_idx, 40)
            current_tauu = tauu_schedule.get(round_idx, 0.1)
            print(f"\n===== 迭代轮次 {round_idx+1}/{iter_rounds} =====") 
            print(f"epoch数目：{epochs} | η={current_eta} | τu={current_tauu}")
            # 应用上一轮的基因权重对数据加权（第一轮除外）
            if round_idx > 0 and prev_gene_weights is not None:
                # 验证权重形状匹配
                if prev_gene_weights.shape[0] != expr_binned_train.shape[1]:
                    raise ValueError(f"基因权重形状({prev_gene_weights.shape})与训练数据基因数({expr_binned_train.shape[1]})不匹配！")
                # 应用权重加权
                expr_binned_train_weighted = expr_binned_train * prev_gene_weights
                expr_binned_val_weighted = expr_binned_val * prev_gene_weights
                print(f"✅ 应用上一轮基因权重加权数据：权重均值={prev_gene_weights.mean():.4f}，权重范围=[{prev_gene_weights.min():.4f}, {prev_gene_weights.max():.4f}]")
                # 数据校验
                check_nan_inf(expr_binned_train_weighted, "expr_binned_train_weighted", f"轮次{round_idx}训练集加权分bin矩阵")
                check_nan_inf(expr_binned_val_weighted, "expr_binned_val_weighted", f"轮次{round_idx}验证集加权分bin矩阵")
            else:
                # 第一轮使用原始数据
                expr_binned_train_weighted = expr_binned_train.copy()
                expr_binned_val_weighted = expr_binned_val.copy()
                print(f"✅ 第一轮训练，使用原始分bin矩阵，未加权")           
            # 构建数据集（修改为使用加权后的数据）
            age_group_train = adata_train.obs[age_group_col].values
            age_group_val = adata_val.obs[age_group_col].values
            train_dataset = AgeRegressionDataset(
                expr_binned=expr_binned_train_weighted,  # 使用加权后的数据
                age_scaled=train_age_scaled,
                cell_prior_u=cell_prior_u_train,
                age_group=age_group_train
            )
            val_dataset = AgeRegressionDataset(
                expr_binned=expr_binned_val_weighted,  # 使用加权后的数据
                age_scaled=val_age_scaled,
                cell_prior_u=cell_prior_u_val,
                age_group=age_group_val
            )
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
            # 初始化模型
            current_n_ct_genes = adata_train_non_a.shape[1]
            age_max_train = np.nanmax(adata_train.obs["age_numeric"].values)
            model = AgeRegressionAE(
                n_genes=current_n_ct_genes, 
                embed_dim=embed_dim,
                n_bins=n_bins,
                age_max_train=age_max_train,
                prior_gene_weights=ct_gene_contribution.values 
            ).to(DEVICE)
            # 加载上一轮参数
            if prev_state_dict_path and os.path.exists(prev_state_dict_path):
                prev_state_dict = torch.load(prev_state_dict_path, map_location=DEVICE)
                model = load_model_params_safely(model, prev_state_dict, device=DEVICE)
                # 从加载的模型中提取基因权重（防止prev_gene_weights为空）
                if prev_gene_weights is None:
                    current_gene_weights = model.gene_weight.detach().cpu().numpy() * current_eta
                    weight_processor.add_round_weights(current_eta, current_tauu, current_gene_weights)
                    prev_gene_weights = current_gene_weights.copy()
                    print(f"✅ 从轮次{round_idx-1}的检查点加载基因权重：均值={current_gene_weights.mean():.4f}")
            # 优化器与调度器
            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=3, 
                 min_lr=1e-7 )  # verbose=True,
            # 训练配置
            model.train()
            patience = np.ceil(epochs/2)
            patience_counter = 0
            best_val_total_loss_round = float('inf')
            best_epoch = 0
            # 记录本轮模型文件
            current_best_model_path = os.path.join(BASE_MODEL_PATH, f"best_model_round_{round_idx}.pth")
            generated_pth_files.append(current_best_model_path)
            torch.save(model.state_dict(), current_best_model_path)
            print(f"✅ 初始化最佳模型（兜底）：{current_best_model_path}")
            # 训练循环
            for epoch in tqdm(range(epochs), desc=f"轮次{round_idx+1} - Epoch"):
                # 训练阶段
                train_total_loss = 0.0
                train_gene_quantile_loss = 0.0
                train_age_reg_loss = 0.0
                train_reg_loss = 0.0
                model.train()                
                for batch_data in train_loader:
                    expr_tensor = batch_data[0]
                    age_scaled_tensor = batch_data[1]
                    cell_u_tensor = batch_data[2]
                    age_group_batch = batch_data[3]                    
                    expr = expr_tensor.to(DEVICE)
                    age_scaled = age_scaled_tensor.to(DEVICE)
                    cell_u = cell_u_tensor.to(DEVICE)                    
                    stage_true_batch = torch.tensor(
                        get_age_stage_label_by_group(age_group_batch),
                        device=DEVICE
                    )                    
                    optimizer.zero_grad()                    
                    # 前向传播
                    latent, gene_quantile_out, age_pred, stage_probs = model(expr, cell_u)                    
                    # 损失计算：分位数损失使用模型内部权重
                    gene_quantile_loss_val = quantile_loss(expr, gene_quantile_out, quantile_levels, torch.sigmoid(model.gene_weight))
                    age_total_loss, reg_loss, stage_loss = age_regression_loss(
                        age_scaled, age_pred, stage_true_batch, stage_probs, age_max_train=age_max_train
                    )
                    current_tauu = tauu_schedule.get(round_idx, 0.1)
                    lambda_u = max(0.01, current_tauu)
                    reg_loss_val = u_prior_regularization_loss(latent, cell_u, DEVICE)                   
                    weight_l1_loss = 0.001 * torch.norm(torch.sigmoid(model.gene_weight), p=1)
                    total_loss = 0.2 * gene_quantile_loss_val + 0.7 * age_total_loss + 0.1 * lambda_u * reg_loss_val + weight_l1_loss
                    total_loss = torch.nan_to_num(total_loss, nan=1.0, posinf=10.0, neginf=0.0)                    
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    total_loss.backward()
                    optimizer.step()                  
                    train_total_loss += total_loss.item()
                    train_gene_quantile_loss += gene_quantile_loss_val.item()
                    train_age_reg_loss += age_total_loss.item()
                    train_reg_loss += reg_loss_val.item()           
                # 验证阶段
                val_total_loss = 0.0
                val_gene_quantile_loss = 0.0
                val_age_reg_loss = 0.0
                model.eval()
                with torch.no_grad():
                    for batch_data in val_loader:
                        expr = batch_data[0].to(DEVICE)
                        age_scaled = batch_data[1].to(DEVICE)
                        cell_u = batch_data[2].to(DEVICE)
                        age_group_batch = batch_data[3]                        
                        stage_true_batch = torch.tensor(
                            get_age_stage_label_by_group(age_group_batch),
                            device=DEVICE
                        )                        
                        latent, gene_quantile_out, age_pred, stage_probs = model(expr, cell_u)                        
                        gene_quantile_loss_val = quantile_loss(expr, gene_quantile_out, quantile_levels, torch.sigmoid(model.gene_weight))
                        age_reg_loss_val, reg_loss_detail, stage_loss_detail = age_regression_loss(age_scaled, age_pred, stage_true_batch, stage_probs, age_max_train=age_max_train)
                        current_tauu = tauu_schedule.get(round_idx, 0.1)
                        lambda_u = max(0.01, current_tauu)
                        reg_loss_val = u_prior_regularization_loss(latent, cell_u, DEVICE)                        
                        total_loss = 0.3 * gene_quantile_loss_val + 0.7 * age_reg_loss_val + lambda_u * reg_loss_val  ####### 改参数
                        total_loss = torch.nan_to_num(total_loss, nan=1.0, posinf=10.0, neginf=0.0)                        
                        val_total_loss += total_loss.item()
                        val_gene_quantile_loss += gene_quantile_loss_val.item()
                        val_age_reg_loss += age_reg_loss_val.item()                
                avg_train_loss = train_total_loss / len(train_loader)
                avg_train_gene_q = train_gene_quantile_loss / len(train_loader)
                avg_train_age_r = train_age_reg_loss / len(train_loader)
                avg_val_loss = val_total_loss / len(val_loader)
                avg_val_gene_q = val_gene_quantile_loss / len(val_loader)
                avg_val_age_r = val_age_reg_loss / len(val_loader)                
                scheduler.step(avg_val_loss)                
                print(f"Epoch {epoch+1}/{epochs} | "
                      f"训练总损失: {avg_train_loss:.4f} | 训练ct_genes分位数损失: {avg_train_gene_q:.4f} | 训练年龄回归损失: {avg_train_age_r:.4f} | "
                      f"验证总损失: {avg_val_loss:.4f} | 验证ct_genes分位数损失: {avg_val_gene_q:.4f} | 验证年龄回归损失: {avg_val_age_r:.4f}")                
                # 早停逻辑
                if avg_val_age_r < best_val_total_loss_round - 1e-5:
                    best_val_total_loss_round = avg_val_age_r
                    best_epoch = epoch + 1                    
                    # 保存当前轮的最佳模型
                    current_best_model_path = os.path.join(BASE_MODEL_PATH, f"best_model_round_{round_idx}.pth")
                    torch.save(model.state_dict(), current_best_model_path)                    
                    # 检查是否为全局最佳
                    if avg_val_age_r < best_global_val_loss:
                        best_global_val_loss = avg_val_age_r
                        best_global_round_idx = round_idx
                        best_global_model_path = os.path.join(BASE_MODEL_PATH, "best_model_global.pth")
                        torch.save(model.state_dict(), best_global_model_path)
                        if best_global_model_path not in generated_pth_files:
                            generated_pth_files.append(best_global_model_path)
                        print(f"🔥 更新全局最佳模型：轮次{round_idx+1} - 第{best_epoch}epoch，验证损失={avg_val_loss:.4f}")
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(f"⚠️  早停触发（验证总损失连续{patience}轮未下降）")
                        if os.path.exists(current_best_model_path):
                            model.load_state_dict(torch.load(current_best_model_path, map_location=DEVICE))
                            print(f"✅ 加载最佳模型：{current_best_model_path}")
                        else:
                            print(f"⚠️  最佳模型文件不存在，使用当前模型继续训练")
                        break            
            # 提取当前轮的基因权重，用于下一轮加权
            current_gene_weights = model.gene_weight.detach().cpu().numpy() * current_eta
            weight_processor.add_round_weights(current_eta, current_tauu, current_gene_weights) 
            prev_gene_weights = current_gene_weights.copy()
            print(f"✅ 保存轮次{round_idx}的基因权重：均值={current_gene_weights.mean():.4f}，范围=[{current_gene_weights.min():.4f}, {current_gene_weights.max():.4f}]")           
            # 保存检查点
            checkpoint_path = os.path.join(BASE_MODEL_PATH, f"age_regression_checkpoint_round_{round_idx}.pth")
            generated_pth_files.append(checkpoint_path)
            torch.save(model.state_dict(), checkpoint_path)            
            # 删除上一轮冗余文件
            if round_idx > 0:
                prev_checkpoint = os.path.join(BASE_MODEL_PATH, f"age_regression_checkpoint_round_{round_idx-1}.pth")
                if os.path.exists(prev_checkpoint):
                    os.remove(prev_checkpoint)
                    generated_pth_files.remove(prev_checkpoint)
                    print(f"🗑️  删除上一轮检查点：{prev_checkpoint}")
                prev_best_model = os.path.join(BASE_MODEL_PATH, f"best_model_round_{round_idx-1}.pth")
                if os.path.exists(prev_best_model):
                    os.remove(prev_best_model)
                    generated_pth_files.remove(prev_best_model)
                    print(f"删除上一轮最佳模型：{prev_best_model}")            
            # 清理
            del optimizer, scheduler
            torch.cuda.empty_cache()
            gc.collect()            
            # ========== nnNU训练 ==========
            print("=== 二分类nnNU训练（A组=未衰老，非A组=衰老）===")
            model.eval()
            # 构建全量数据集（使用原始分箱矩阵）
            expr_binned_full = adata_train.layers[layer_name].copy()
            cell_prior_u_full = compute_sample_level_prior_u(
                adata_train, ct_gene_contribution, "age_numeric", 
                tauu_factor=tauu_schedule.get(round_idx+1, 0.1)
            )
            full_dataset = AgeRegressionDataset(
                expr_binned=expr_binned_full,
                age_scaled=adata_train.obs["age_scaled"].values,
                cell_prior_u=cell_prior_u_full,
                age_group=adata_train.obs[age_group_col].values
            )
            full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)
            # 提取latent
            all_latents = []
            with torch.no_grad():
                for batch_data in full_loader:
                    expr = batch_data[0].to(DEVICE)
                    cell_u = batch_data[2].to(DEVICE)
                    latent, _, _, _ = model(expr, cell_u)
                    all_latents.append(latent)
            latents_torch = torch.cat(all_latents, dim=0)
            # 构建掩码
            control_mask = adata_train.obs[age_group_col] == "A"
            is_non_control_mask_torch = torch.tensor(~control_mask, dtype=torch.bool, device=DEVICE)
            control_mask_torch = torch.tensor(control_mask, dtype=torch.bool, device=DEVICE)
            control_latents_torch = latents_torch[control_mask_torch]
            # 初始化nnNU分类器
            nnnu_classifier = BinarynnNUClassifier(
                input_dim=embed_dim,
                hidden_dim=nnnu_params["hidden_dim"],
                dropout=nnnu_params["dropout"]
            ).to(DEVICE)
            nnnu_optimizer = optim.AdamW(
                nnnu_classifier.parameters(),
                lr=nnnu_params["lr"],
                weight_decay=nnnu_params["weight_decay"]
            )
            # nnNU训练（alpha迭代）
            alpha_start = nnnu_params["alpha_start"]
            alpha_decrement = nnnu_params["alpha_decrement"]
            alpha_min = 0.05
            alpha = alpha_start - round_idx * alpha_decrement  # 初始alpha
            best_calibration = 1.0
            optimal_alpha = alpha
            alpha_pbar = tqdm(
                total=int(np.ceil((alpha_start - alpha_min) / alpha_decrement)) + 1,
                desc="nnNU-alpha迭代校准（二分类）",
                leave=True,
                position=0,
                ncols=100
            )
            while alpha >= alpha_min:
                alpha_pbar.set_postfix({
                    "当前alpha": round(alpha, 2),
                    "最佳校准值": f"{best_calibration:.3f}"
                })
                nnnu_classifier.train()
                train_pbar = tqdm(
                    range(nnnu_params["train_rounds"]),
                    desc=f"nnNU训练 (alpha={round(alpha,2)})",
                    leave=False,
                    position=1,
                    ncols=100
                )
                for _ in train_pbar:
                    nnnu_optimizer.zero_grad()
                    shuffle_idx = torch.randperm(len(latents_torch), device=DEVICE)
                    latents_shuffle = latents_torch[shuffle_idx]
                    is_non_control_shuffle = is_non_control_mask_torch[shuffle_idx]
                    g = nnnu_classifier(latents_shuffle)
                    risk = binary_nnNU_risk(g, is_non_control_shuffle, alpha)
                    risk.backward()
                    nnnu_optimizer.step()
                    train_pbar.set_postfix({"风险值": f"{risk.item():.4f}"})
                    del shuffle_idx, latents_shuffle, is_non_control_shuffle, g, risk
                    torch.cuda.empty_cache()
                train_pbar.close() 
                # 校准
                nnnu_classifier.eval()
                with torch.no_grad():
                    control_g = nnnu_classifier(control_latents_torch)
                    control_g = torch.nan_to_num(control_g, nan=0.0, posinf=10.0, neginf=-10.0)
                    control_g = torch.clamp(control_g, -100.0, 100.0)
                    control_posterior = torch.sigmoid(control_g)
                    control_posterior = torch.clamp(control_posterior, 0.0, 1.0) 
                    calib_mean = torch.mean(control_posterior).cpu().numpy()
                    calib_mean = np.nan_to_num(calib_mean, nan=0.0, posinf=1.0, neginf=0.0)          
                # 更新最佳校准
                if calib_mean <= nnnu_params["calibration_target"] and calib_mean < best_calibration:
                    best_calibration = calib_mean
                    optimal_alpha = alpha
                    print(f"✅ 校准达标（{calib_mean:.3f} ≤ {nnnu_params['calibration_target']}），提前终止alpha迭代")
                    alpha_pbar.update(alpha_pbar.total - alpha_pbar.n + 1) 
                    break   
                # alpha衰减
                alpha -= alpha_decrement
                alpha = max(alpha, alpha_min)
                alpha_pbar.update(1)
            alpha_pbar.close()
            print(f"✅ 二分类nnNU校准完成：alpha={round(optimal_alpha,2)}，A组平均衰老概率={best_calibration:.3f}")
            # 清理nnNU相关
            del nnnu_classifier, nnnu_optimizer, latents_torch, control_latents_torch
            torch.cuda.empty_cache()
            gc.collect()
        # ========== 训练结束，保存结果 ==========
        print("\n=== 训练完成，准备保存必要文件 ===")
        if best_global_model_path is None:
            best_global_model_path = os.path.join(BASE_MODEL_PATH, f"best_model_round_{iter_rounds-1}.pth")
            best_global_round_idx = iter_rounds - 1
            print(f"⚠️  未找到全局最佳模型，使用最后一轮模型：{best_global_model_path}")
        else:
            print(f"✅ 全局最佳模型：{best_global_model_path}")
            print(f"  来自轮次 {best_global_round_idx+1}/{iter_rounds}，验证损失={best_global_val_loss:.4f}")
        # 根据全局最佳轮次截断权重处理器
        print(f"\n📊 根据全局最佳轮次 {best_global_round_idx+1} 保存权重...")
        truncated_weight_processor = weight_processor.truncate_to_round(best_global_round_idx)
        print(f"  原始权重轮次数: {len(weight_processor.weight_rounds)}")
        print(f"  截断后轮次数: {len(truncated_weight_processor.weight_rounds)}")
        weight_processor_path = os.path.join(BASE_MODEL_PATH, "weight_processor.pkl")
        truncated_weight_processor.save(weight_processor_path)
        print(f"✅ 保存截断后的多轮权重处理器到：{weight_processor_path}")
        # 同时保存完整权重处理器用于调试
        full_weight_processor_path = os.path.join(BASE_MODEL_PATH, "weight_processor_full.pkl")
        weight_processor.save(full_weight_processor_path)
        print(f"📁 同时保存完整权重处理器到：{full_weight_processor_path}（用于调试）")
        # 保存贡献度信息
        ct_gene_contribution.to_pickle(ct_gene_contribution_path)
        # 保存全局最佳轮次信息到文本文件
        round_info_path = os.path.join(BASE_MODEL_PATH, "best_round_info.txt")
        with open(round_info_path, 'w') as f:
            f.write(f"best_round={best_global_round_idx}\n")
            f.write(f"best_val_loss={best_global_val_loss:.6f}\n")
            f.write(f"best_model_path={best_global_model_path}\n")
            f.write(f"weight_processor_rounds={len(truncated_weight_processor.weight_rounds)}\n")
        print(f"\n🎯 训练完成总结:") # 
        print(f"  最佳模型: {os.path.basename(best_global_model_path)}")
        print(f"  最佳轮次: {best_global_round_idx + 1}")
        print(f"  验证损失: {best_global_val_loss:.4f}")
        # 清理临时文件
        final_keep_files = [
            best_global_model_path, 
            weight_processor_path,
            os.path.join(BASE_MODEL_PATH, "ct_gene_contribution.pkl"),
        ]
        for file_path in generated_pth_files:
            if file_path not in final_keep_files and os.path.exists(file_path):
                os.remove(file_path)
                print(f"🗑️  清理临时文件：{file_path}") 
    except Exception as e:
        print(f"\n❌ 训练异常：{type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        raise
    finally:
        torch.cuda.empty_cache()
        gc.collect()
    return best_global_model_path, full_weight_processor_path, ct_gene_contribution_path, round_info_path



# ===================== 预测函数（直接使用原始分箱矩阵）=====================
def predict_age(model_path, adata, 
                weight_processor_full_path,    # 使用完整处理器路径
                best_round_info_txt_path,      # 最佳轮次信息文本文件路径
                embed_dim=128, layer_name=LAYER_NAME, device=None,
                age_range_fixed=None, n_bins=32, ct_gene_contribution_path=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # 年龄缩放
    print(f"\n=== 年龄数据缩放处理 ===")
    if age_range_fixed is not None:
        age_max_scale = age_range_fixed[1]
        print(f"使用手动指定的年龄最大值：{age_max_scale}")
    else:
        age_max_scale = np.nanmax(adata.obs["age_numeric"].values)
        print(f"使用数据年龄最大值：{age_max_scale}")
    adata.obs["age_scaled"] = adata.obs["age_numeric"] / age_max_scale
    print(f"缩放后年龄范围：[{adata.obs['age_scaled'].min():.4f}, {adata.obs['age_scaled'].max():.4f}]")
    print(f"\n=== 加载模型并预测年龄 ===")
    # 1. 加载完整权重处理器
    weight_processor = MultiRoundWeightProcessor.load(weight_processor_full_path)
    print(f"✅ 加载完整权重处理器，共{len(weight_processor.weight_rounds)}轮")
    print(f"   基因权重列表长度：{len(weight_processor.gene_weight_rounds)}")
    # 2. 从文本文件读取最佳轮次索引
    best_round_idx = None
    with open(best_round_info_txt_path, 'r') as f:
        for line in f:
            if line.startswith('best_round='):
                best_round_idx = int(line.strip().split('=')[1])
                break
    if best_round_idx is None:
        raise ValueError(f"无法从文件中解析 best_round: {best_round_info_txt_path}")
    print(f"✅ 最佳轮次索引：{best_round_idx}")
    # 3. 获取该轮次的 tauu 因子
    best_tauu = weight_processor.get_round_tauu_factor(round_idx=best_round_idx)
    print(f"✅ 最佳轮次 tauu 因子：{best_tauu}")
    # 4. 加载贡献度，计算先验 u 值
    ct_gene_contribution = pd.read_pickle(ct_gene_contribution_path)
    # 计算先验 u 值
    best_tauu = weight_processor.get_round_tauu_factor(best_round_idx)
    cell_prior_u = compute_sample_level_prior_u(adata, ct_gene_contribution, "age_numeric",
                                                tauu_factor=best_tauu, round_idx=best_round_idx)
    # 应用加权（修正索引）
    expr_binned = adata.layers[layer_name].copy()
    expr_binned_weighted = weight_processor.apply_single_round_weighting(expr_binned, round_idx=best_round_idx, n_bins=n_bins)
    n_genes = adata.shape[1]
    model = AgeRegressionAE(
        n_genes=n_genes,
        embed_dim=embed_dim,
        n_bins=n_bins,
        age_max_train=age_max_scale,
        prior_gene_weights=ct_gene_contribution.values
    ).to(device)
    # 7. 加载模型权重
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    # 8. 构建数据集
    dataset = AgeRegressionDataset(
        expr_binned=expr_binned_weighted,
        age_scaled=adata.obs["age_scaled"].values,
        cell_prior_u=cell_prior_u
    )
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
    # 9. 预测循环
    all_age_scaled_pred = []
    all_latents = []
    all_stage_probs = []
    with torch.no_grad():
        for batch_data in tqdm(dataloader, desc="预测年龄"):
            expr = batch_data[0].to(device)
            cell_u_batch = batch_data[2].to(device)
            latent, _, age_scaled_pred, stage_probs = model(expr, cell_u_batch)
            all_age_scaled_pred.append(age_scaled_pred.cpu().numpy())
            all_latents.append(latent.cpu().numpy())
            all_stage_probs.append(stage_probs.cpu().numpy())
    age_scaled_pred = np.concatenate(all_age_scaled_pred, axis=0)
    age_predict = age_scaled_pred * age_max_scale
    stage_probs = np.concatenate(all_stage_probs, axis=0)
    stage_pred_label = np.argmax(stage_probs, axis=1)
    # 鲁棒裁剪
    age_min = np.nanmin(adata.obs["age_numeric"].values)
    age_max = np.nanmax(adata.obs["age_numeric"].values)
    age_predict = np.clip(age_predict, max(0, age_min * 0.5), age_max * 1.2)
    # 保存结果
    adata.obs["age_predict"] = age_predict
    adata.obsm["latent_features"] = np.concatenate(all_latents, axis=0)
    adata.obs["age_stage_pred"] = stage_pred_label
    print(f"\n✅ 融合模型预测完成：")
    print(f"  缩放预测年龄范围：{age_scaled_pred.min():.4f} ~ {age_scaled_pred.max():.4f}")
    print(f"  实际预测年龄范围：{age_predict.min():.2f} ~ {age_predict.max():.2f}")
    print(f"  数据实际年龄范围：{age_min:.2f} ~ {age_max:.2f}")
    return adata, age_predict

# -------------------------- 绘图逻辑适配 --------------------------
def plot_age_prediction(y_train_true, y_train_pred, y_test_true, y_test_pred, 
                        save_path=None, title="Age Prediction (Regression AE)"):
    """重构：同时可视化训练集和测试集的预测结果"""
    # 初始化画布
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    # 定义颜色
    train_color = "#87B5D2"  # 训练集浅蓝色
    test_color = "#A23B72"   # 测试集紫色
    perfect_line_color = 'r'
    # 计算评估指标
    train_mae = mean_absolute_error(y_train_true, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
    train_r2 = r2_score(y_train_true, y_train_pred)
    train_pearson = pearsonr(y_train_true, y_train_pred)[0]
    train_spearman = spearmanr(y_train_true, y_train_pred)[0]
    test_mae = mean_absolute_error(y_test_true, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
    test_r2 = r2_score(y_test_true, y_test_pred)
    test_pearson = pearsonr(y_test_true, y_test_pred)[0]
    test_spearman = spearmanr(y_test_true, y_test_pred)[0]
    # 绘制散点图
    ax.scatter(y_train_true, y_train_pred, alpha=0.4, s=40, color=train_color, 
               edgecolors='white', linewidth=0.5, label=f"Training Set (n={len(y_train_true)})")
    ax.scatter(y_test_true, y_test_pred, alpha=0.6, s=50, color=test_color, 
               edgecolors='white', linewidth=0.5, label=f"Test Set (n={len(y_test_true)})")
    # 完美预测线
    all_true = np.concatenate([y_train_true, y_test_true])
    all_pred = np.concatenate([y_train_pred, y_test_pred])
    min_age = min(all_true.min(), all_pred.min())
    max_age = max(all_true.max(), all_pred.max())
    ax.plot([min_age, max_age], [min_age, max_age], 
            linestyle='--', linewidth=2, color=perfect_line_color, 
            label="Perfect Prediction (y=x)")
    # 标注评估指标（两列）
    metric_text = (
        f"Training Set (n={len(y_train_true)}):\n"
        f"MAE = {train_mae:.2f}\n"
        f"RMSE = {train_rmse:.2f}\n"
        f"R² = {train_r2:.3f}\n"
        f"Pearson r = {train_pearson:.3f}\n"
        f"Spearman r = {train_spearman:.3f}\n\n"
        f"Test Set (n={len(y_test_true)}):\n"
        f"MAE = {test_mae:.2f}\n"
        f"RMSE = {test_rmse:.2f}\n"
        f"R² = {test_r2:.3f}\n"
        f"Pearson r = {test_pearson:.3f}\n"
        f"Spearman r = {test_spearman:.3f}"
    ) 
    ax.text(min_age + (max_age - min_age) * 0.05, 
            max_age - (max_age - min_age) * 0.2, 
            metric_text, 
            fontsize=9, 
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.8),
            verticalalignment='top',
            family='monospace')
    # 图属性
    ax.set_xlabel("True Age", fontsize=12)
    ax.set_ylabel("Predicted Age", fontsize=12)
    ax.grid(alpha=0.3, linestyle='-', linewidth=0.5)
    ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
    plt.tight_layout()
    # 保存
    if save_path is None:
        plt.show(fig)
    else:
        save_path_full = os.path.join(save_path, 'age_prediction_regression.png')
        plt.savefig(save_path_full, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"✅ 年龄预测图已保存至：{save_path_full}")

        

In [ ]:


# 步骤3：构建样本-细胞类型均值adata（核心重构）
adata_mean = build_sample_celltype_mean_adata(adata, age_group_col=AGE_GROUP_COL)
gc.collect()

# 步骤4：移除低质量样本（AE训练前）
print("\n" + "="*50)
print("步骤4：移除低质量样本")
print("="*50)
adata_high_quality = filter_low_quality_samples(adata_mean)
del adata_mean
gc.collect()

# 步骤5：筛选DEG基因
print("\n" + "="*50)
print("步骤6：筛选Age_group相关DEG基因")
print("="*50)
adata_deg_only = get_age_group_degs(adata_high_quality, age_group_col=AGE_GROUP_COL, n_top_degs=5000)  ### 改参数
del adata_high_quality
gc.collect()

# 步骤6：分bin处理
print("\n" + "="*50)
print("步骤7：全局分bin处理")
print("="*50)
adata_deg_binned = bin_expression_matrix(adata_deg_only)
del adata_deg_only
gc.collect()


bin_save_path = os.path.join(BASE_DATA_PATH, 'Endo_healthy_bin_noI.h5ad')
adata_deg_binned = sc.read_h5ad(bin_save_path)
# 步骤7：样本级别划分训练/测试集（排除A组）
print("\n" + "="*50)
print("步骤5：划分训练/测试集（排除A组）")
print("="*50)
adata_train, adata_val, adata_test = split_sample_train_test(
    adata_deg_binned, test_size=0.2, random_state=4 , age_group_col=AGE_GROUP_COL)

# 步骤8：训练年龄回归AE（仅用测试集）
print("\n" + "="*50)
print("步骤8：训练年龄回归AE模型") 
print("="*50)
model_path, weight_processor_path, ct_gene_contribution_path, round_info_path = train_age_regression_ae(
    adata_train=adata_train,
    adata_val=adata_val,
    embed_dim=EMBED_DIM, 
    epochs=EPOCHS,  # 1
    iter_rounds=ITER_ROUNDS,  # 
    batch_size=BATCH_SIZE, # 
    layer_name=LAYER_NAME,
    age_group_col=AGE_GROUP_COL,
    age_range_fixed=AGE_RANGE_FIXED
)
del adata_deg_binned
gc.collect()

# 步骤9：预测年龄
adata_test, test_age_predict = predict_age(
    model_path=model_path, 
    adata=adata_test,
    weight_processor_full_path = weight_processor_path,
    best_round_info_txt_path = round_info_path,
    ct_gene_contribution_path = ct_gene_contribution_path,
    embed_dim=EMBED_DIM,
    layer_name=LAYER_NAME,
    age_range_fixed=AGE_RANGE_FIXED,
)

# 预测训练集 
print("预测训练集年龄...")
adata_train, train_age_predict = predict_age(
    model_path = model_path,
    adata = adata_train,
    weight_processor_full_path = weight_processor_path,
    best_round_info_txt_path = round_info_path,
    ct_gene_contribution_path = ct_gene_contribution_path,
    embed_dim = EMBED_DIM,
    layer_name =LAYER_NAME,
    age_range_fixed=AGE_RANGE_FIXED
)

adata_train.write(os.path.join(BASE_DATA_PATH, "Endo_healthy_train_final.h5ad"))
adata_test.write(os.path.join(BASE_DATA_PATH, "Endo_healthy_test_final.h5ad"))

# 步骤10：可视化预测结果（排除A组）
print("\n" + "="*50)
print("步骤10：可视化预测结果")
print("="*50)

# 排除A组
train_non_a_mask = adata_train.obs[AGE_GROUP_COL] != "A"
test_non_a_mask = adata_test.obs[AGE_GROUP_COL] != "A"

y_train_true = adata_train.obs.loc[train_non_a_mask, "age_numeric"].values
y_train_pred = adata_train.obs.loc[train_non_a_mask, "age_predict"].values
y_test_true = adata_test.obs.loc[test_non_a_mask, "age_numeric"].values
y_test_pred = adata_test.obs.loc[test_non_a_mask, "age_predict"].values

print(f"训练集（非A组）：{len(y_train_true)}个样本")
print(f"测试集（非A组）：{len(y_test_true)}个样本")

# 可视化整体结果
plot_age_prediction(
    y_train_true=y_train_true,
    y_train_pred=y_train_pred,
    y_test_true=y_test_true,
    y_test_pred=y_test_pred,
    save_path=BASE_PLOT_PATH,
    title="Age Prediction (Regression AE - Exclude A Group)"
)





In [ ]:
len(adata_train.obs['sample'].unique()),len(adata_test.obs['sample'].unique()),len(adata_val.obs['sample'].unique())


In [ ]:
adata.obs['Celltype'].value_counts()


In [ ]:
def safe_read_h5ad(h5_path):
    data_dict = {}
    with h5py.File(h5_path, "r") as f:
        # 读取X，兼容稀疏COO存储
        if "X" in f:
            if "data" in f["X"]:
                # COO稀疏矩阵
                data = f["X"]["data"][:]
                i = f["X"]["indices"][:]
                j = f["X"]["indptr"][:]
                shape = tuple(f["X"]["shape"][:])
                data_dict["X"] = csr_matrix((data, i, j), shape=shape)
            else:
                data_dict["X"] = np.array(f["X"][:])
        # obs 表格
        obs = {}
        for col in f["obs"].keys():
            ds = f["obs"][col]
            arr = ds[:]
            if ds.dtype.kind == "S":
                arr = [s.decode("utf8") for s in arr]
            obs[col] = arr
        data_dict["obs"] = pd.DataFrame(obs)
        # var 表格
        var = {}
        for col in f["var"].keys():
            ds = f["var"][col]
            arr = ds[:]
            if ds.dtype.kind == "S":
                arr = [s.decode("utf8") for s in arr]
            var[col] = arr
        data_dict["var"] = pd.DataFrame(var)
        # 修复layers分组（核心报错点，兼容稀疏矩阵）
        data_dict["layers"] = {}
        if "layers" in f:
            for lay_name in f["layers"].keys():
                lay_group = f["layers"][lay_name]
                if "data" in lay_group:
                    data = lay_group["data"][:]
                    idx = lay_group["indices"][:]
                    ptr = lay_group["indptr"][:]
                    shp = tuple(lay_group["shape"][:])
                    mat = csr_matrix((data, idx, ptr), shape=shp)
                else:
                    mat = np.array(lay_group[:])
                data_dict["layers"][lay_name] = mat
        # obsm/varm稠密嵌入
        data_dict["obsm"] = {}
        if "obsm" in f:
            for k in f["obsm"].keys():
                data_dict["obsm"][k] = np.array(f["obsm"][k][:])
        data_dict["varm"] = {}
        if "varm" in f:
            for k in f["varm"].keys():
                data_dict["varm"][k] = np.array(f["varm"][k][:])
    return data_dict


In [ ]:
# sorted(adata_mean)
adata_ref



In [ ]:
adata_ref = adata
# 步骤3：构建样本-细胞类型均值adata（核心重构）
adata_mean = build_sample_celltype_mean_adata(adata, age_group_col=AGE_GROUP_COL)
gc.collect()


In [ ]:
# 步骤4：移除低质量样本（AE训练前）
print("\n" + "="*50)
print("步骤4：移除低质量样本")
print("="*50)
adata_high_quality = filter_low_quality_samples(adata_mean)
del adata_mean
gc.collect()


In [ ]:
# 步骤5：筛选DEG基因
print("\n" + "="*50)
print("步骤6：筛选Age_group相关DEG基因")
print("="*50)
adata_deg_only = get_age_group_degs(adata_high_quality, age_group_col=AGE_GROUP_COL, n_top_degs=10000)
del adata_high_quality
gc.collect()

# 步骤6：分bin处理
print("\n" + "="*50)
print("步骤7：全局分bin处理")
print("="*50)
adata_deg_binned = bin_expression_matrix(adata_deg_only)
del adata_deg_only
gc.collect()


In [ ]:
# sorted(dir())


In [ ]:
# 步骤7：样本级别划分训练/测试集（排除A组）
print("\n" + "="*50)
print("步骤5：划分训练/测试集（排除A组）")
print("="*50)
adata_train, adata_val, adata_test = split_sample_train_test(
    adata_deg_binned, test_size=0.2, random_state=4 , age_group_col=AGE_GROUP_COL)

# 步骤8：训练年龄回归AE（仅用测试集）
print("\n" + "="*50)
print("步骤8：训练年龄回归AE模型") 
print("="*50)
model_path, weight_processor_path, ct_gene_contribution_path, round_info_path = train_age_regression_ae(
    adata_train=adata_train,
    ### 加 损失权重分配; Status; get_age_stage_label_by_group + age regression (0.1,0.9); Quantile loss+ REG+先验 (0.3, 0.7,先验*lamda); HVGs; Age_Scale
    adata_val=adata_val,
    embed_dim=EMBED_DIM, 
    epochs=EPOCHS,  # 1
    iter_rounds=ITER_ROUNDS,  # 
    batch_size=BATCH_SIZE, # 
    layer_name=LAYER_NAME,
    age_group_col=AGE_GROUP_COL,
    age_range_fixed=AGE_RANGE_FIXED
)
del adata_deg_binned
gc.collect()


In [ ]:
# 步骤9：预测年龄
adata_test, test_age_predict = predict_age(
    model_path=model_path, 
    adata=adata_test,
    weight_processor_full_path = weight_processor_path,
    best_round_info_txt_path = round_info_path,
    ct_gene_contribution_path = ct_gene_contribution_path,
    embed_dim=EMBED_DIM,
    layer_name=LAYER_NAME,
    age_range_fixed=AGE_RANGE_FIXED,   
)


In [ ]:
# 预测训练集 
print("预测训练集年龄...")
adata_train, train_age_predict = predict_age(
    model_path = model_path,
    adata = adata_train,
    weight_processor_full_path = weight_processor_path,
    best_round_info_txt_path = round_info_path,
    ct_gene_contribution_path = ct_gene_contribution_path,
    embed_dim = EMBED_DIM,
    layer_name =LAYER_NAME,
    age_range_fixed=AGE_RANGE_FIXED
)

adata_train.write(os.path.join(BASE_DATA_PATH, "Fibro_healthy_train_final.h5ad"))
adata_test.write(os.path.join(BASE_DATA_PATH, "Fibro_healthy_test_final.h5ad"))

# 步骤10：可视化预测结果（排除A组）
print("\n" + "="*50)
print("步骤10：可视化预测结果")
print("="*50)


In [ ]:
adata_test.shape, adata_val.shape, adata_train.shape


In [ ]:
adata_test.obs[['age_numeric', 'age_scaled']].drop_duplicates()


In [ ]:
# 预测验证集  
print("预测验证集年龄...")
adata_train, train_age_predict = predict_age(
    model_path = model_path,
    adata = adata_val,
    weight_processor_full_path = weight_processor_path,
    best_round_info_txt_path = round_info_path,
    ct_gene_contribution_path = ct_gene_contribution_path,
    embed_dim = EMBED_DIM,
    layer_name =LAYER_NAME,
    age_range_fixed=AGE_RANGE_FIXED
)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

def plot_age_prediction(y_train_true, y_train_pred, 
                        y_val_true, y_val_pred, 
                        y_test_true, y_test_pred, 
                        save_path=None, title="Age Prediction (Regression AE)"):
    """
    同时绘制训练集、验证集、测试集的真实年龄 vs 预测年龄散点图，并显示各集评价指标。

    参数:
        y_train_true, y_train_pred : 训练集真实值和预测值 (array-like)
        y_val_true, y_val_pred     : 验证集真实值和预测值 (array-like)
        y_test_true, y_test_pred   : 测试集真实值和预测值 (array-like)
        save_path : 保存路径（目录），若为 None 则显示图形
        title : 图表主标题
    """
    # 合并所有真实值用于坐标轴范围
    all_true = np.concatenate([y_train_true, y_val_true, y_test_true])
    all_pred = np.concatenate([y_train_pred, y_val_pred, y_test_pred])
    min_age = min(all_true.min(), all_pred.min()) - 2
    max_age = max(all_true.max(), all_pred.max()) + 2

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    # 颜色定义
    train_color = "#87B5D2"   # 浅蓝
    val_color   = "#F4A261"   # 橙黄
    test_color  = "#A23B72"   # 紫

    # ---------- 散点 ----------
    ax.scatter(y_train_true, y_train_pred, alpha=0.4, s=40, color=train_color,
               edgecolors='white', linewidth=0.5, label=f"Training (n={len(y_train_true)})")
    ax.scatter(y_val_true, y_val_pred, alpha=0.6, s=40, color=val_color,
               edgecolors='white', linewidth=0.5, label=f"Validation (n={len(y_val_true)})")
    ax.scatter(y_test_true, y_test_pred, alpha=0.6, s=50, color=test_color,
               edgecolors='white', linewidth=0.5, label=f"Test (n={len(y_test_true)})")

    # ---------- 完美预测线 ----------
    ax.plot([min_age, max_age], [min_age, max_age],
            linestyle='--', linewidth=2, color='red', label="Perfect Prediction (y=x)")

    # ---------- 计算各集指标 ----------
    def compute_metrics(true, pred):
        mae = mean_absolute_error(true, pred)
        rmse = np.sqrt(mean_squared_error(true, pred))
        r2 = r2_score(true, pred)
        pearson = pearsonr(true, pred)[0]
        spearman = spearmanr(true, pred)[0]
        return mae, rmse, r2, pearson, spearman

    train_metrics = compute_metrics(y_train_true, y_train_pred)
    val_metrics   = compute_metrics(y_val_true, y_val_pred)
    test_metrics  = compute_metrics(y_test_true, y_test_pred)

    # ---------- 指标文本（分三列显示） ----------
    metric_text = (
        f"{'Training':<15} {'Validation':<15} {'Test':<15}\n"
        f"{'MAE':<5} {train_metrics[0]:<6.2f}    {val_metrics[0]:<6.2f}    {test_metrics[0]:<6.2f}\n"
        f"{'RMSE':<5} {train_metrics[1]:<6.2f}    {val_metrics[1]:<6.2f}    {test_metrics[1]:<6.2f}\n"
        f"{'R²':<5}  {train_metrics[2]:<6.3f}    {val_metrics[2]:<6.3f}    {test_metrics[2]:<6.3f}\n"
        f"{'Pearson':<5} {train_metrics[3]:<6.3f}    {val_metrics[3]:<6.3f}    {test_metrics[3]:<6.3f}\n"
        f"{'Spearman':<5} {train_metrics[4]:<6.3f}    {val_metrics[4]:<6.3f}    {test_metrics[4]:<6.3f}"
    )

    ax.text(min_age + (max_age - min_age) * 0.05,
            max_age - (max_age - min_age) * 0.25,
            metric_text,
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.8),
            verticalalignment='top',
            family='monospace')

    # ---------- 图属性 ----------
    ax.set_xlabel("True Age", fontsize=12)
    ax.set_ylabel("Predicted Age", fontsize=12)
    ax.set_xlim(min_age, max_age)
    ax.set_ylim(min_age, max_age)
    ax.grid(alpha=0.3, linestyle='-', linewidth=0.5)
    ax.legend(loc="lower right", fontsize=10, framealpha=0.9)

    plt.tight_layout()

    # ---------- 保存或显示 ----------
    if save_path is None:
        plt.show()
    else:
        save_path_full = os.path.join(save_path, 'age_prediction_with_val.png')
        plt.savefig(save_path_full, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"✅ 年龄预测图已保存至：{save_path_full}")
        

In [ ]:
# 步骤10：可视化预测结果（排除A组）

# 排除A组
train_non_a_mask = adata_train.obs[AGE_GROUP_COL] != "A"
test_non_a_mask = adata_test.obs[AGE_GROUP_COL] != "A"
val_non_a_mask = adata_val.obs[AGE_GROUP_COL] != "A"
y_train_true = adata_train.obs.loc[train_non_a_mask, "age_numeric"].values
y_train_pred = adata_train.obs.loc[train_non_a_mask, "age_predict"].values
y_test_true = adata_test.obs.loc[test_non_a_mask, "age_numeric"].values
y_test_pred = adata_test.obs.loc[test_non_a_mask, "age_predict"].values
y_val_true = adata_val.obs.loc[val_non_a_mask, "age_numeric"].values
y_val_pred = adata_val.obs.loc[val_non_a_mask, "age_predict"].values

print(f"训练集（非A组）：{len(y_train_true)}个样本")
print(f"测试集（非A组）：{len(y_test_true)}个样本")
print(f"测试集（非A组）：{len(y_val_true)}个样本")


In [ ]:
# sorted(dir())
# dir()


In [ ]:
# 步骤10：可视化预测结果（排除A组）

# 排除A组
train_non_a_mask = adata_train.obs["age_numeric"] > 1
test_non_a_mask = adata_test.obs["age_numeric"] > 1

y_train_true = adata_train.obs.loc[train_non_a_mask, "age_numeric"].values
y_train_pred = adata_train.obs.loc[train_non_a_mask, "age_predict"].values
y_test_true = adata_test.obs.loc[test_non_a_mask, "age_numeric"].values
y_test_pred = adata_test.obs.loc[test_non_a_mask, "age_predict"].values

print(f"训练集（非A组）：{len(y_train_true)}个样本")
print(f"测试集（非A组）：{len(y_test_true)}个样本")

# 可视化整体结果
plot_age_prediction(
    y_train_true=y_train_true,
    y_train_pred=y_train_pred,
    y_test_true=y_test_true,
    y_test_pred=y_test_pred,
    save_path=BASE_PLOT_PATH,
    title="Age Prediction (Regression AE - above 1y )"
)



In [ ]:
plot_age_prediction(
    y_train_true=y_train_true,
    y_train_pred=y_train_pred,
    y_val_true=y_val_true,
    y_val_pred=y_val_pred,
    y_test_true=y_test_true,
    y_test_pred=y_test_pred,
    save_path=BASE_PLOT_PATH,
    title="Age Prediction (Regression AE - Exclude A Group)"
)


In [ ]:

# 包括 A 组
y_train_true = adata_train.obs["age_numeric"].values
y_train_pred = adata_train.obs["age_predict"].values
y_test_true = adata_test.obs[ "age_numeric"].values
y_test_pred = adata_test.obs[ "age_predict"].values

print(f"训练集（非A组）：{len(y_train_true)}个样本")
print(f"测试集（非A组）：{len(y_test_true)}个样本")

# 可视化整体结果
plot_age_prediction(
    y_train_true=y_train_true,
    y_train_pred=y_train_pred,
    y_test_true=y_test_true,
    y_test_pred=y_test_pred,
    save_path=BASE_PLOT_PATH,
    title="Age Prediction (Regression AE - include A Group)"
)


In [ ]:
train_non_a_mask.value_counts()

In [ ]:
len(adata_test.obs['sample'].unique())


In [ ]:
# 步骤9：预测外验数据的 Age , 要先
adata_external = sc.read_h5ad('external_inputs/Fibro_valid_normal_age_inteWithTrainA.h5ad')



In [ ]:
# 步骤3：构建样本-细胞类型均值adata（核心重构）
adata_mean_valid = build_sample_celltype_mean_adata(adata_external, age_group_col=AGE_GROUP_COL)
gc.collect()

# 步骤4：移除低质量样本（AE训练前）

adata_high_quality_valid = filter_low_quality_samples(adata_mean_valid)
del adata_mean_valid
gc.collect()
adata_deg_only = get_age_group_degs(adata_high_quality_valid, age_group_col=AGE_GROUP_COL, n_top_degs=10000)
del adata_high_quality_valid
gc.collect()

# 步骤6：分bin处理
print("\n" + "="*50)
print("步骤7：全局分bin处理")
print("="*50)
adata_deg_binned_valid = bin_expression_matrix(adata_deg_only)
del adata_deg_only
gc.collect()



In [ ]:
# 步骤9：预测年龄, 
adata_test, test_age_predict = predict_age(
    model_path=model_path, 
    adata=adata_deg_binned_valid,
    weight_processor_full_path = weight_processor_path,
    best_round_info_txt_path = round_info_path,
    ct_gene_contribution_path = ct_gene_contribution_path,
    embed_dim=EMBED_DIM,
    layer_name=LAYER_NAME,
    age_range_fixed=AGE_RANGE_FIXED,
)     ## 设计，或者加一个后面，可以随意预测外验数据; 



In [ ]:
AGE_GROUP_COL


In [ ]:
# GateFusion
import scanpy as sc 


In [ ]:
########### Below is the RF celltype proportion + gene expression code 

for h5_path in normal_adata_dir_list:
    print(f"\nProcessing: {h5_path}")
    adata = sc.read_h5ad(h5_path)
    print(h5_path)
    for K_EXPR_FEATURES in k_features_list:
        print(K_EXPR_FEATURES)
        # Ensure Age is numeric
        if adata.obs['Age'].dtype.name == 'category' or adata.obs['Age'].dtype == 'object':
            adata.obs['Age'] = adata.obs['Age'].astype(float)
        # Determine type (with 'A' group or not)
        type_label = 'A' if 'A' in adata.obs['Age_group'].unique() else 'noA'
        # Extract lineage name from path
        if '/Fibro_' in h5_path:
            lineage = 'fibro'
        else:
            lineage = 'endo'
        # Get target cell types present in this dataset
        target_cell_types = sorted(adata.obs['Celltype'].unique())
        print(f"Cell types: {target_cell_types}")
        # Build full feature matrix (proportions + expression)
        print("Building feature matrix...")
        df_all = build_feature_matrix(adata, target_cell_types)
        print(f"Feature matrix shape: {df_all.shape}")
        # Separate target and features
        y = df_all['Age'].values
        X = df_all.drop(columns='Age')
        # Split into proportion and expression features
        prop_cols = [col for col in X.columns if col.endswith('_ratio')]
        expr_cols = [col for col in X.columns if col not in prop_cols]
        X_prop = X[prop_cols]
        X_expr = X[expr_cols]
        print(f"Proportion features: {X_prop.shape[1]}, Expression features: {X_expr.shape[1]}")
        # Feature selection on expression part (to avoid extreme dimensionality)
        if X_expr.shape[1] > 0:
            X_expr_selected, selected_expr_cols, vt, skb = select_expression_features(
                X_expr, y, k_features=K_EXPR_FEATURES, variance_threshold=VARIANCE_THRESH
            )
            print(f"Selected {len(selected_expr_cols)} expression features out of {X_expr.shape[1]}")
        else:
            X_expr_selected = np.empty((X_prop.shape[0], 0)) 
            selected_expr_cols = []
        
        # Combine proportion features (no selection) and selected expression features
        X_final = np.hstack([X_prop.values, X_expr_selected])
        all_feature_names = list(prop_cols) + list(selected_expr_cols)
        print(f"Final feature matrix: {X_final.shape}")
        # Train Random Forest with CV
        print("Training Random Forest with hyperparameter tuning...")
        best_rf, y_cv_pred, best_params = rf_cv_with_features(X_final, y, use_loocv=USE_LOOCV, random_state=42, n_iter=30)
        print(f"Best parameters: {best_params}")
        # Evaluate
        output_dir = os.path.join(base_output_dir, lineage, type_label,str(K_EXPR_FEATURES))
        os.makedirs(output_dir, exist_ok=True)
        prefix = f"{lineage}_{type_label}"
        metrics = evaluate_and_save(y, y_cv_pred, best_rf, all_feature_names, output_dir, prefix,
                                    extra_info={'nPropFeat': len(prop_cols), 'nExprFeat': len(selected_expr_cols)})
        # Save metrics to list
        evaluation_rows.append({
            'Celltype': lineage,
            'Type': type_label,
            'nPropFeat': len(prop_cols),
            'nExprFeat': len(selected_expr_cols),
            'R2': metrics['R2'],
            'MAE': metrics['MAE'],
            'MAD': metrics['MAD'],
            'MSE': metrics['MSE'],
            'RMSE': metrics['RMSE'],
            'PCC': metrics['PCC'],
            'Pval': metrics['Pval']
        })
    # Free memory
    del adata, df_all, X_prop, X_expr, X_final, best_rf
    clear_memory()

# ======================== Process combined datasets (Fibro + Endo) ========================
for h5_path_pair in normal_mergeadata_dir_list:
    print(f"\nProcessing combined: {h5_path_pair}")
    adata_fib = sc.read_h5ad(h5_path_pair[0])
    adata_endo = sc.read_h5ad(h5_path_pair[1])
    # Ensure Age numeric for endo
    if adata_endo.obs['Age'].dtype.name == 'category' or adata_endo.obs['Age'].dtype == 'object':
        adata_endo.obs['Age'] = adata_endo.obs['Age'].astype(float)
    adata = sc.concat([adata_fib, adata_endo])
    del adata_fib, adata_endo
    for K_EXPR_FEATURES in k_features_list: 
        type_label = 'A' if 'A' in adata.obs['Age_group'].unique() else 'noA'
        lineage = 'FibEndo'
        target_cell_types = sorted(adata.obs['Celltype'].unique())
        print(f"Cell types in combined: {len(target_cell_types)}")
        # Build feature matrix
        print("Building feature matrix for combined data...")
        df_all = build_feature_matrix(adata, target_cell_types)
        print(f"Feature matrix shape: {df_all.shape}")
        y = df_all['Age'].values
        X = df_all.drop(columns='Age')
        prop_cols = [col for col in X.columns if col.endswith('_ratio')]
        expr_cols = [col for col in X.columns if col not in prop_cols]
        X_prop = X[prop_cols]
        X_expr = X[expr_cols]
        print(f"Proportion features: {X_prop.shape[1]}, Expression features: {X_expr.shape[1]}")
        if X_expr.shape[1] > 0:
            X_expr_selected, selected_expr_cols, vt, skb = select_expression_features(
                X_expr, y, k_features=K_EXPR_FEATURES, variance_threshold=VARIANCE_THRESH
            )
            print(f"Selected {len(selected_expr_cols)} expression features")
        else:
            X_expr_selected = np.empty((X_prop.shape[0], 0))
            selected_expr_cols = []
        
        X_final = np.hstack([X_prop.values, X_expr_selected])
        all_feature_names = list(prop_cols) + list(selected_expr_cols)
        print(f"Final feature matrix: {X_final.shape}")
        print("Training Random Forest on combined data...")
        best_rf, y_cv_pred, best_params = rf_cv_with_features(X_final, y, use_loocv=USE_LOOCV, random_state=42, n_iter=30)
        print(f"Best parameters: {best_params}")
        output_dir = os.path.join(base_output_dir, lineage, type_label,str(K_EXPR_FEATURES))
        os.makedirs(output_dir, exist_ok=True)
        prefix = f"{lineage}_{type_label}"
        metrics = evaluate_and_save(y, y_cv_pred, best_rf, all_feature_names, output_dir, prefix,
                                    extra_info={'nPropFeat': len(prop_cols), 'nExprFeat': len(selected_expr_cols)})
        evaluation_rows.append({
            'Celltype': lineage,
            'Type': type_label,
            'nPropFeat': len(prop_cols),
            'nExprFeat': len(selected_expr_cols),
            'R2': metrics['R2'],
            'MAE': metrics['MAE'],
            'MAD': metrics['MAD'],
            'MSE': metrics['MSE'],
            'RMSE': metrics['RMSE'],
            'PCC': metrics['PCC'],
            'Pval': metrics['Pval']
        })
    del adata, df_all, X_final, best_rf
    clear_memory()

# Save benchmark table
evaluation_df = pd.DataFrame(evaluation_rows, columns=columns_list)
output_tsv = os.path.join(base_output_dir, 'RF_ExprProp_benchmarking.tsv')
evaluation_df.to_csv(output_tsv, sep='\t', index=False)
print(f"\nBenchmarking results saved to: {output_tsv}")
print(evaluation_df)


In [ ]:
normal_adata_dir_list = ['/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib_age_normal.h5ad',
 '/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib_age_normal_delA.h5ad',
'/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo_age_normal.h5ad',
'/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo_age_normal_delA.h5ad'                                   ]



In [ ]:
normal_mergeadata_dir_list = [
        [ '/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib_age_normal.h5ad',
            '/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo_age_normal.h5ad'],
        [ '/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib_age_normal_delA.h5ad',
            '/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo_age_normal_delA.h5ad' ]
    ]
